In [4]:
import pandas as pd

df = pd.read_excel('data/PoliceStationLocation.xlsx')
df.head()

,시도경찰청,위치,경찰서명칭,경찰서주소
0,서울특별시경찰청,서울특별시,서울중부경찰서,중구 수표로 27
1,서울특별시경찰청,서울특별시,서울종로경찰서,종로구 율곡로 46
2,서울특별시경찰청,서울특별시,서울남대문경찰서,중구 한강대로 410
3,서울특별시경찰청,서울특별시,서울서대문경찰서,서대문구 통일로 113
4,서울특별시경찰청,서울특별시,서울혜화경찰서,종로구 창경궁로 112-16


In [6]:
df = pd.read_excel("data/PoliceStationLocation.xlsx")
print(df.columns.tolist())  # 실제 컬럼 이름 확인
print(df.head(3))

['시도경찰청', '위치', '경찰서명칭', '경찰서주소']
      시도경찰청     위치     경찰서명칭        경찰서주소
0  서울특별시경찰청  서울특별시   서울중부경찰서    중구 수표로 27
1  서울특별시경찰청  서울특별시   서울종로경찰서   종로구 율곡로 46
2  서울특별시경찰청  서울특별시  서울남대문경찰서  중구 한강대로 410


In [9]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
params = {"query": "서울시 강남구 테헤란로 1"}

res = requests.get(url, headers=headers, params=params)
print(res.status_code)
print(res.json())

403
{'errorType': 'NotAuthorizedError', 'message': 'App(홍진우) disabled OPEN_MAP_AND_LOCAL service.'}


In [11]:
import requests
import pandas as pd
import time
from tqdm import tqdm  # pip install tqdm

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address: str) -> tuple[float | None, float | None]:
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    try:
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        res.raise_for_status()
        docs = res.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])
    except Exception as e:
        print(f"에러: {address} → {e}")
    return None, None


def geocode_dataframe(df: pd.DataFrame, address_col: str, save_path: str = "output.xlsx") -> pd.DataFrame:
    """
    - 진행상황 표시 (tqdm)
    - 실패한 주소 재시도 1회
    - 1,000건마다 중간 저장 (중단돼도 이어서 가능)
    """
    # 이미 처리된 결과 있으면 이어서 시작
    try:
        done = pd.read_excel(save_path)
        processed = set(done[address_col].tolist())
        print(f"이어서 시작: {len(done)}건 완료됨")
    except FileNotFoundError:
        done = pd.DataFrame()
        processed = set()

    remaining = df[~df[address_col].isin(processed)].copy()
    results = []

    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining))):
        addr = getattr(row, address_col)
        lat, lng = geocode(addr)

        # 실패 시 1회 재시도
        if lat is None:
            time.sleep(1)
            lat, lng = geocode(addr)

        results.append({**row._asdict(), "lat": lat, "lng": lng})
        time.sleep(0.05)  # 초당 20건 이하로 유지

        # 1,000건마다 중간 백업 (CSV로 안전하게)
        if (i + 1) % 1000 == 0:
            chunk = pd.DataFrame(results).drop(columns=["Index"], errors="ignore")
            combined = pd.concat([done, chunk], ignore_index=True)
            combined.to_csv("temp_backup.csv", index=False, encoding="utf-8-sig")
            print(f"\n중간 백업 완료: {len(combined)}건 (temp_backup.csv)")

    # 최종 저장 (엑셀)
    final_df = pd.DataFrame(results).drop(columns=["Index"], errors="ignore")
    combined = pd.concat([done, final_df], ignore_index=True)
    combined.to_excel(save_path, index=False)

    # 실패한 주소 리포트 (엑셀)
    failed = combined[combined["lat"].isna()]
    if len(failed) > 0:
        print(f"\n⚠️ 변환 실패: {len(failed)}건")
        failed[[address_col]].to_excel("failed_addresses.xlsx", index=False)

    print(f"\n✅ 완료: 총 {len(combined)}건 → {save_path}")
    return combined


# 사용 예시
df = pd.read_excel("data/PoliceStationLocation.xlsx")   # 엑셀 파일 읽기
result = geocode_dataframe(df, address_col="경찰서주소", save_path="geocoded2.xlsx")

100%|██████████████████████████████████████████████████████████████████████████████████| 59/59 [00:11<00:00,  5.09it/s]


⚠️ 변환 실패: 4건

✅ 완료: 총 59건 → geocoded2.xlsx


In [12]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]["y"]), float(docs[0]["x"])
    return None, None

addresses = [
    "종로구 율곡로 46",
    "강서구 화곡로 73",
    "성북구 종암로 135",
    "구로구 가마산로 235"
]

for addr in addresses:
    lat, lng = geocode(addr)
    print(f"{addr} → 위도: {lat}, 경도: {lng}")

종로구 율곡로 46 → 위도: None, 경도: None
강서구 화곡로 73 → 위도: None, 경도: None
성북구 종암로 135 → 위도: None, 경도: None
구로구 가마산로 235 → 위도: None, 경도: None


In [14]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]["y"]), float(docs[0]["x"])
    return None, None

addresses = [
    "서울특별시 종로구 율곡로 46",
    "서울특별시 강서구 화곡로 73",
    "서울특별시 성북구 종암로 135",
    "서울특별시 구로구 가마산로 235"
]

for addr in addresses:
    lat, lng = geocode(addr)
    print(f"{addr} → 위도: {lat}, 경도: {lng}")

서울특별시 종로구 율곡로 46 → 위도: None, 경도: None
서울특별시 강서구 화곡로 73 → 위도: None, 경도: None
서울특별시 성북구 종암로 135 → 위도: None, 경도: None
서울특별시 구로구 가마산로 235 → 위도: None, 경도: None


In [15]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
res = requests.get(url, headers=headers, params={"query": "서울특별시 종로구 율곡로 46"}, timeout=5)

print(res.status_code)
print(res.json())

200
{'documents': [], 'meta': {'is_end': True, 'pageable_count': 0, 'total_count': 0}}


In [16]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"  # 여기 바뀜
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]["y"]), float(docs[0]["x"])
    return None, None

addresses = [
    "서울특별시 종로구 율곡로 46",
    "서울특별시 강서구 화곡로 73",
    "서울특별시 성북구 종암로 135",
    "서울특별시 구로구 가마산로 235"
]

for addr in addresses:
    lat, lng = geocode(addr)
    print(f"{addr} → 위도: {lat}, 경도: {lng}")

서울특별시 종로구 율곡로 46 → 위도: None, 경도: None
서울특별시 강서구 화곡로 73 → 위도: None, 경도: None
서울특별시 성북구 종암로 135 → 위도: None, 경도: None
서울특별시 구로구 가마산로 235 → 위도: None, 경도: None


In [17]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    # 앞에 "서울특별시 " 자동으로 붙이기
    full_address = "서울특별시 " + address
    res = requests.get(url, headers=headers, params={"query": full_address}, timeout=5)
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]["y"]), float(docs[0]["x"])
    return None, None

addresses = [
    "종로구 율곡로 46",
    "강서구 화곡로 73",
    "성북구 종암로 135",
    "구로구 가마산로 235"
]

for addr in addresses:
    lat, lng = geocode(addr)
    print(f"{addr} → 위도: {lat}, 경도: {lng}")

종로구 율곡로 46 → 위도: None, 경도: None
강서구 화곡로 73 → 위도: None, 경도: None
성북구 종암로 135 → 위도: None, 경도: None
구로구 가마산로 235 → 위도: None, 경도: None


In [18]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

url = "https://dapi.kakao.com/v2/local/search/keyword.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
res = requests.get(url, headers=headers, params={"query": "서울특별시 종로구 율곡로 46"}, timeout=5)

print(res.status_code)
print(res.json())

200
{'documents': [], 'meta': {'is_end': True, 'pageable_count': 0, 'same_name': {'keyword': '', 'region': [], 'selected_region': '서울 종로구 율곡로'}, 'total_count': 0}}


In [19]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    # 앞에 "서울특별시 " 자동으로 붙이기
    full_address = "서울특별시 " + address
    res = requests.get(url, headers=headers, params={"query": full_address}, timeout=5)
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]["y"]), float(docs[0]["x"])
    return None, None

addresses = [
    "서울 종로구 인사동5길 41",
"서울 강서구 화곡동 980-27",
"서울 성북구 종암동 3-1260",
"구로동 3-25"
]

for addr in addresses:
    lat, lng = geocode(addr)
    print(f"{addr} → 위도: {lat}, 경도: {lng}")

서울 종로구 인사동5길 41 → 위도: None, 경도: None
서울 강서구 화곡동 980-27 → 위도: None, 경도: None
서울 성북구 종암동 3-1260 → 위도: None, 경도: None
구로동 3-25 → 위도: 37.5067164863743, 경도: 126.890496837047


In [20]:
import requests
import pandas as pd
import time
from tqdm import tqdm  # pip install tqdm

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address: str) -> tuple[float | None, float | None]:
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    try:
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        res.raise_for_status()
        docs = res.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])
    except Exception as e:
        print(f"에러: {address} → {e}")
    return None, None


def geocode_dataframe(df: pd.DataFrame, address_col: str, save_path: str = "output.xlsx") -> pd.DataFrame:
    """
    - 진행상황 표시 (tqdm)
    - 실패한 주소 재시도 1회
    - 1,000건마다 중간 저장 (중단돼도 이어서 가능)
    """
    # 이미 처리된 결과 있으면 이어서 시작
    try:
        done = pd.read_excel(save_path)
        processed = set(done[address_col].tolist())
        print(f"이어서 시작: {len(done)}건 완료됨")
    except FileNotFoundError:
        done = pd.DataFrame()
        processed = set()

    remaining = df[~df[address_col].isin(processed)].copy()
    results = []

    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining))):
        addr = getattr(row, address_col)
        lat, lng = geocode(addr)

        # 실패 시 1회 재시도
        if lat is None:
            time.sleep(1)
            lat, lng = geocode(addr)

        results.append({**row._asdict(), "lat": lat, "lng": lng})
        time.sleep(0.05)  # 초당 20건 이하로 유지

        # 1,000건마다 중간 백업 (CSV로 안전하게)
        if (i + 1) % 1000 == 0:
            chunk = pd.DataFrame(results).drop(columns=["Index"], errors="ignore")
            combined = pd.concat([done, chunk], ignore_index=True)
            combined.to_csv("temp_backup.csv", index=False, encoding="utf-8-sig")
            print(f"\n중간 백업 완료: {len(combined)}건 (temp_backup.csv)")

    # 최종 저장 (엑셀)
    final_df = pd.DataFrame(results).drop(columns=["Index"], errors="ignore")
    combined = pd.concat([done, final_df], ignore_index=True)
    combined.to_excel(save_path, index=False)

    # 실패한 주소 리포트 (엑셀)
    failed = combined[combined["lat"].isna()]
    if len(failed) > 0:
        print(f"\n⚠️ 변환 실패: {len(failed)}건")
        failed[[address_col]].to_excel("failed_addresses2.xlsx", index=False)

    print(f"\n✅ 완료: 총 {len(combined)}건 → {save_path}")
    return combined


# 사용 예시
df = pd.read_excel("data/PoliceLocation.xlsx")   # 엑셀 파일 읽기
result = geocode_dataframe(df, address_col="주소", save_path="geocoded3.xlsx")

100%|████████████████████████████████████████████████████████████████████████████████| 461/461 [01:03<00:00,  7.21it/s]



⚠️ 변환 실패: 9건

✅ 완료: 총 461건 → geocoded3.xlsx


In [23]:
import pandas as pd

df = pd.read_excel('data/alarmbell.xlsx')
df.head()

,안전비상벨관리번호,설치목적,설치장소유형,설치위치,소재지도로명주소,소재지지번주소,WGS84위도,WGS84경도,연계방식,경찰연계유무,경비업체연계유무,관리사무소연계유무,부가기능,안전비상벨설치연도,최종점검일자,최종점검결과구분,관리기관명,관리기관전화번호,데이터기준일자
0,건물-6호,방범용,건물,서초동 1549-4,"서울특별시 서초구 반포대로22길 39, 우신1549빌딩 (서초동)",서울특별시 서초구 서초동 1549-4 우신1549빌딩,37.48990,127.01067,미연계,N,N,N,NaN,2019,2019-07-02,Y,서초구청,02-2155-6830,2020-10-30
1,건물-5호,방범용,건물,방배동 426-10,서울특별시 서초구 방배천로16길 11-5 (방배동),서울특별시 서초구 방배동 437-10,37.48242,126.98335,미연계,N,N,N,NaN,2019,2019-07-09,Y,서초구청,02-2155-6830,2020-10-30
2,건물-7호,방범용,건물,서초동 1481-22,서울특별시 서초구 반포대로5길 54 (서초동),서울특별시 서초구 서초동 1481-22,37.48168,127.00865,미연계,N,N,N,NaN,2019,2019-09-20,Y,서초구청,02-2155-6830,2020-10-30
3,건물-9호,방범용,건물,서초동 1481-1,"서울특별시 서초구 반포대로9길 57, 서초그레이스빌 (서초동)",서울특별시 서초구 서초동 1481-1 서초그레이스빌,37.48287,127.00912,미연계,N,N,N,NaN,2019,2019-10-31,Y,서초구청,02-2155-6830,2020-10-30
4,건물-10호,방범용,건물,서초동 1481-21,"서울특별시 서초구 반포대로9길 55, 힐하우스 (서초동)",서울특별시 서초구 서초동 1481-21 힐하우스,37.48275,127.00854,미연계,N,N,N,NaN,2019,2019-10-31,Y,서초구청,02-2155-6830,2020-10-30


In [37]:
df1 = pd.read_excel('data/alarmbell.xlsx')


df1 = df1.rename(columns={'WGS84위도':'위도', 'WGS84경도': '경도'})
df1

df1 = df1[['설치위치', '소재지지번주소', '위도', '경도']]
df1

df.to_excel('data/alarm.xlsx',index=False)
print("저장완료")

저장완료


,안전비상벨관리번호,설치위치,위도,경도
0,건물-6호,서초동 1549-4,37.489900,127.010670
1,건물-5호,방배동 426-10,37.482420,126.983350
2,건물-7호,서초동 1481-22,37.481680,127.008650
3,건물-9호,서초동 1481-1,37.482870,127.009120
4,건물-10호,서초동 1481-21,37.482750,127.008540
...,...,...,...,...
47922,김포-A-169,주택가·도로·골목길·공원·어린이보호구역 등,37.614316,126.713202
47923,김포-A-170,주택가·도로·골목길·공원·어린이보호구역 등,37.622300,126.722610
47924,김포-A-171,주택가·도로·골목길·공원·어린이보호구역 등,37.619407,126.714704
47925,김포-A-172,주택가·도로·골목길·공원·어린이보호구역 등,37.622593,126.694730


KeyError: "['소재지지번주 소'] not in index"

['안전비상벨관리번호', '설치위치', '위도', '경도']


In [4]:
import pandas as pd
df=pd.read_excel('data/유흥업소정제거의.xlsx')
df

,Unnamed: 0,사업장명,지번주소
0,0,성진,서울특별시 종로구 청진동 26
1,1,한일,서울특별시 종로구 묘동 200
2,2,슬로우(SLOW),서울특별시 종로구 돈의동 19 (지하1층)
3,3,큐노리방,서울특별시 종로구 창신동 465
4,4,슈퍼맨,서울특별시 종로구 종로5가 245
...,...,...,...
4498,4498,상상노래클럽,"경기도 김포시 구래동 6882-5 이너매스한강 (511,512)호"
4499,4499,좋은느낌노래짱,경기도 김포시 구래동 6883-8 다온프라자 203호
4500,4500,반창꼬뮤직뱅크,경기도 김포시 구래동 6883-8 다온프라자 303호
4501,4501,설레임노래타운,경기도 김포시 구래동 6883-8 다온프라자 304~305호


In [5]:
import requests
import pandas as pd
import time
from tqdm import tqdm  # pip install tqdm

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def geocode(address: str) -> tuple[float | None, float | None]:
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    try:
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        res.raise_for_status()
        docs = res.json().get("documents", [])
        if docs:
            return float(docs[0]["y"]), float(docs[0]["x"])
    except Exception as e:
        print(f"에러: {address} → {e}")
    return None, None


def geocode_dataframe(df: pd.DataFrame, address_col: str, save_path: str = "output.xlsx") -> pd.DataFrame:
    """
    - 진행상황 표시 (tqdm)
    - 실패한 주소 재시도 1회
    - 1,000건마다 중간 저장 (중단돼도 이어서 가능)
    """
    # 이미 처리된 결과 있으면 이어서 시작
    try:
        done = pd.read_excel(save_path)
        processed = set(done[address_col].tolist())
        print(f"이어서 시작: {len(done)}건 완료됨")
    except FileNotFoundError:
        done = pd.DataFrame()
        processed = set()

    remaining = df[~df[address_col].isin(processed)].copy()
    results = []

    for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining))):
        addr = getattr(row, address_col)
        lat, lng = geocode(addr)

        # 실패 시 1회 재시도
        if lat is None:
            time.sleep(1)
            lat, lng = geocode(addr)

        results.append({**row._asdict(), "lat": lat, "lng": lng})
        time.sleep(0.05)  # 초당 20건 이하로 유지

        # 1,000건마다 중간 백업 (CSV로 안전하게)
        if (i + 1) % 1000 == 0:
            chunk = pd.DataFrame(results).drop(columns=["Index"], errors="ignore")
            combined = pd.concat([done, chunk], ignore_index=True)
            combined.to_csv("temp_backup.csv", index=False, encoding="utf-8-sig")
            print(f"\n중간 백업 완료: {len(combined)}건 (temp_backup.csv)")

    # 최종 저장 (엑셀)
    final_df = pd.DataFrame(results).drop(columns=["Index"], errors="ignore")
    combined = pd.concat([done, final_df], ignore_index=True)
    combined.to_excel(save_path, index=False)

    # 실패한 주소 리포트 (엑셀)
    failed = combined[combined["lat"].isna()]
    if len(failed) > 0:
        print(f"\n⚠️ 변환 실패: {len(failed)}건")
        failed[[address_col]].to_excel("failed_addresses2.xlsx", index=False)

    print(f"\n✅ 완료: 총 {len(combined)}건 → {save_path}")
    return combined


# 사용 예시
df = pd.read_excel('data/유흥업소정제거의.xlsx')   # 엑셀 파일 읽기
result = geocode_dataframe(df, address_col="지번주소", save_path="geocoded3.xlsx")

 22%|█████████████████▎                                                            | 1001/4503 [02:37<07:27,  7.83it/s]


중간 백업 완료: 1000건 (temp_backup.csv)


 44%|██████████████████████████████████▋                                           | 2001/4503 [05:02<05:25,  7.70it/s]


중간 백업 완료: 2000건 (temp_backup.csv)


 67%|███████████████████████████████████████████████████▉                          | 3001/4503 [08:03<03:19,  7.52it/s]


중간 백업 완료: 3000건 (temp_backup.csv)


 89%|█████████████████████████████████████████████████████████████████████▎        | 4000/4503 [10:47<02:06,  3.97it/s]


중간 백업 완료: 4000건 (temp_backup.csv)


100%|██████████████████████████████████████████████████████████████████████████████| 4503/4503 [12:22<00:00,  6.07it/s]



⚠️ 변환 실패: 180건

✅ 완료: 총 4503건 → geocoded3.xlsx


In [3]:
import requests
import pandas as pd

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_dong(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return docs[0]['address']['region_3depth_name']
    return None

df = pd.read_excel('어린이집기본정보서울.xlsx')
df['동'] = df['주소'].apply(get_dong)
df.to_excel('결과.xlsx', index=False)
print('완료!')

FileNotFoundError: [Errno 2] No such file or directory: '어린이집기본정보서울.xlsx'

In [4]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'data', 'failed_addresses.xlsx', 'failed_addresses2.xlsx', 'geocoded1.xlsx', 'geocoded3.xlsx', 'temp_backup.csv', 'Untitled.ipynb', '위경도.ipynb']


In [5]:
print(os.listdir('data'))


['alarmbell.xlsx', 'Caraccident.xlsx', 'caraccident2.xlsx', 'cctv.xlsx', 'crime.xlsx', 'PoliceLocation.xlsx', 'policelocation1.xlsx', 'policelocation2.xlsx', 'PoliceStationLocation.xlsx', '어린이집기본정보서울.xls', '유흥업소정제거의.xlsx']


In [8]:
import requests
import pandas as pd

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_dong(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return docs[0]['address']['region_3depth_name']
    return None

df = pd.read_excel('data/어린이집서울.xlsx')
df['동'] = df['주소'].apply(get_dong)
df.to_excel('결과.xlsx', index=False)
print('완료!')

완료!


In [9]:
import requests
import pandas as pd

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_dong(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return docs[0]['address']['region_3depth_name']
    return None

df = pd.read_excel('data/어린이집인천.xlsx')
df['동'] = df['주소'].apply(get_dong)
df.to_excel('결과2.xlsx', index=False)
print('완료!')

완료!


In [10]:
def get_dong(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        print(docs[0])  # 전체 결과 출력
    return None

get_dong("서울특별시 종로구 대학로14길 34")

{'address': {'address_name': '서울 종로구 혜화동 163-3', 'b_code': '1111016900', 'h_code': '1111065000', 'main_address_no': '163', 'mountain_yn': 'N', 'region_1depth_name': '서울', 'region_2depth_name': '종로구', 'region_3depth_h_name': '혜화동', 'region_3depth_name': '혜화동', 'sub_address_no': '3', 'x': '127.002535737835', 'y': '37.5847761690383'}, 'address_name': '서울 종로구 대학로14길 34', 'address_type': 'ROAD_ADDR', 'road_address': {'address_name': '서울 종로구 대학로14길 34', 'building_name': '혜화어린이집', 'main_building_no': '34', 'region_1depth_name': '서울', 'region_2depth_name': '종로구', 'region_3depth_name': '혜화동', 'road_name': '대학로14길', 'sub_building_no': '', 'underground_yn': 'N', 'x': '127.002535737835', 'y': '37.5847761690383', 'zone_no': '03084'}, 'x': '127.002535737835', 'y': '37.5847761690383'}


In [11]:
import requests
import pandas as pd

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_dong(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return docs[0]['address']['region_3depth_h_name']
    return None

df = pd.read_excel('data/어린이집인천.xlsx')
df['동'] = df['주소'].apply(get_dong)
df.to_excel('행정동인천.xlsx', index=False)
print('완료!')

완료!


In [13]:
import requests
import pandas as pd

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": "KakaoAK YOUR_KAKAO_API_KEY"}

def get_address(address):
    try:
        params = {'query': address}
        res = requests.get(url, headers=headers, params=params, timeout=5)
        a = res.json()
        law_dong = a['documents'][0]['address']['region_3depth_name']
        hang_dong = a['documents'][0]['address']['region_3depth_h_name']
        return pd.Series([law_dong, hang_dong])
    except Exception as e:
        print(f"주소 변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None])

# 여기서 엑셀 파일 넣는 거예요
df = pd.read_excel('data/어린이집인천.xlsx')
df[['법정동', '행정동']] = df['주소'].apply(get_address)
df.to_excel('어린이집인천_동추가.xlsx', index=False)
print('완료!')

주소 변환 실패: 인천 중구 운서동 주공아파트12단지 2786-1번지 1205동  / 에러: list index out of range
주소 변환 실패: 인천 중구 북성동2가 12번지  / 에러: list index out of range
주소 변환 실패: 인천 중구 운서동 주공아파트12단지 1205동104호  / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 신도시북로43번길 32 204-101(운서동, 창보밀레시티) / 에러: list index out of range
주소 변환 실패: 인천 중구 운서동 주공아파트12단지 주공12단지 1205동 103호  / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 젓개로 103-13 꽃나무어린이집 / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 눈돌로13번길 10 13번길 13, 1층(운서동) / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 신도시남로 92 1205-103(운서동, 영종주공스카이빌12단지) / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 신도시남로 92 1205-104(운서동, 영종주공스카이빌12단지) / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 신도시남로 92 1205-105(운서동, 영종주공스카이빌12단지) / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 흰바위로 203 216-103 / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 두미포로 244 LH영종A67 어린이집 / 에러: list index out of range
주소 변환 실패: 인천광역시 중구 흰바위로 249 232-101(운서sk뷰스카이시티2차) / 에러: list index out of range
주소 변환 실패: 인천광

In [14]:
import re

def clean_address(address):
    # 괄호 안 내용 제거 (운서동, 창보밀레시티) 이런거
    address = re.sub(r'\(.*?\)', '', address)
    # 동/로/길 + 숫자 까지만 추출
    match = re.search(r'(.+?(?:동|로|길)\s*[\d-]+)', address)
    if match:
        return match.group(1).strip()
    return address.strip()

# 테스트
test = [
    "인천광역시 중구 신도시북로43번길 32 204-101(운서동, 창보밀레시티)",
    "인천광역시 중구 젓개로 103-13 꽃나무어린이집",
    "인천광역시 동구 봉수대로7번길 6 (송림동)",
    "인천 중구 운서동 주공아파트12단지 2786-1번지 1205동"
]

for t in test:
    print(clean_address(t))

인천광역시 중구 신도시북로43
인천광역시 중구 젓개로 103-13
인천광역시 동구 봉수대로7
인천 중구 운서동 주공아파트12단지 2786-1번지 1205동


In [1]:
import requests
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_dong(address):
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        docs = res.json().get("documents", [])
        
        if not docs:
            return pd.Series([None, None, None])
        
        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        
        final_dong = hang_dong if hang_dong else law_dong
        
        return pd.Series([law_dong, hang_dong, final_dong])
    
    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/어린이집인천.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('행정동인천1.xlsx', index=False)
print('완료!')

100%|██████████████████████████████████████████████████████████████████████████████| 3046/3046 [03:43<00:00, 13.61it/s]


완료!


In [2]:
import requests
import pandas as pd
import re
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def clean_address(address):
    # 괄호 안 내용 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 동호수 제거 (숫자-숫자 패턴)
    address = re.sub(r'\d+-\d+$', '', address)
    # 중복 주소 제거 (같은 주소 두번 쓴 경우)
    address = address.split('인천광역시')[1] if address.count('인천광역시') > 1 else address
    address = address.strip()
    return address

def get_dong(address):
    try:
        clean = clean_address(address)
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": clean}, timeout=5)
        docs = res.json().get("documents", [])

        if not docs:
            return pd.Series([None, None, None])

        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        final_dong = hang_dong if hang_dong else law_dong

        return pd.Series([law_dong, hang_dong, final_dong])

    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/어린이집인천.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('어린이집인천_없는동.xlsx', index=False)
print('완료!')

  7%|█████▎                                                                         | 206/3046 [00:15<03:30, 13.48it/s]


KeyboardInterrupt: 

In [5]:
import requests
import pandas as pd
import re
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def clean_address(address):
    # 괄호 안 내용 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 동호수 제거 (숫자-숫자 패턴)
    address = re.sub(r'\d+-\d+$', '', address)
    # 중복 주소 제거 (같은 주소 두번 쓴 경우)
    address = address.split('인천광역시')[1] if address.count('인천광역시') > 1 else address
    address = address.strip()
    return address

def get_dong(address):
    try:
        clean = clean_address(address)
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": clean}, timeout=5)
        docs = res.json().get("documents", [])

        if not docs:
            return pd.Series([None, None, None])

        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        final_dong = hang_dong if hang_dong else law_dong

        return pd.Series([law_dong, hang_dong, final_dong])

    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/인천없는동.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('어린이집인천_없는동.xlsx', index=False)
print('완료!')

KeyError: '주소'

In [6]:
import requests
import pandas as pd
import re
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def clean_address(address):
    # 괄호 안 내용 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 동호수 제거 (숫자-숫자 패턴)
    address = re.sub(r'\d+-\d+$', '', address)
    # 중복 주소 제거 (같은 주소 두번 쓴 경우)
    address = address.split('인천광역시')[1] if address.count('인천광역시') > 1 else address
    address = address.strip()
    return address

def get_dong(address):
    try:
        clean = clean_address(address)
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": clean}, timeout=5)
        docs = res.json().get("documents", [])

        if not docs:
            return pd.Series([None, None, None])

        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        final_dong = hang_dong if hang_dong else law_dong

        return pd.Series([law_dong, hang_dong, final_dong])

    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/인천없는동.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('어린이집인천_없는동.xlsx', index=False)
print('완료!')

KeyError: '주소'

In [7]:
import requests
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_dong(address):
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        docs = res.json().get("documents", [])
        
        if not docs:
            return pd.Series([None, None, None])
        
        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        
        final_dong = hang_dong if hang_dong else law_dong
        
        return pd.Series([law_dong, hang_dong, final_dong])
    
    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/어린이집경기.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('행정동경기1.xlsx', index=False)
print('완료!')

100%|██████████████████████████████████████████████████████████████████████████████| 3454/3454 [04:07<00:00, 13.97it/s]


완료!


In [8]:
import requests
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_dong(address):
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        docs = res.json().get("documents", [])
        
        if not docs:
            return pd.Series([None, None, None])
        
        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        
        final_dong = hang_dong if hang_dong else law_dong
        
        return pd.Series([law_dong, hang_dong, final_dong])
    
    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/어린이집서울.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('행정동서울1.xlsx', index=False)
print('완료!')

100%|██████████████████████████████████████████████████████████████████████████████| 3945/3945 [04:43<00:00, 13.93it/s]


완료!


In [9]:
def clean_address(address):
    # 괄호 안 내용 완전 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 1~2층, 1단지 등 제거
    address = re.sub(r'\s+\d+[~-]\d+층.*', '', address)
    address = re.sub(r'\s+\d+단지.*', '', address)
    address = re.sub(r'\s+\d+-\d+$', '', address)
    address = address.strip()
    return address

# 테스트
test = "경기도 부천시 원미구 계남로 125 1단지 관리동 (중동, 한라마을 주공1,2단지)"
print(clean_address(test))
# → 경기도 부천시 원미구 계남로 125

경기도 부천시 원미구 계남로 125


In [11]:
import requests
import pandas as pd
import re
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def clean_address(address):
    # 괄호 안 내용 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 동호수 제거 (숫자-숫자 패턴)
    address = re.sub(r'\d+-\d+$', '', address)
    # 중복 주소 제거 (같은 주소 두번 쓴 경우)
    address = address.split('인천광역시')[1] if address.count('인천광역시') > 1 else address
    address = address.strip()
    return address

def get_dong(address):
    try:
        clean = clean_address(address)
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": clean}, timeout=5)
        docs = res.json().get("documents", [])

        if not docs:
            return pd.Series([None, None, None])

        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        final_dong = hang_dong if hang_dong else law_dong

        return pd.Series([law_dong, hang_dong, final_dong])

    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/인천없는동.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('어린이집인천_없는동.xlsx', index=False)
print('완료!')

['인천광역시', '중구', '해찬나래어린이집', '인천광역시 중구 두미포로 244 LH영종A67 어린이집']


KeyError: '주소'

In [1]:
import requests
import pandas as pd
import re
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def clean_address(address):
    # 괄호 안 내용 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 동호수 제거 (숫자-숫자 패턴)
    address = re.sub(r'\d+-\d+$', '', address)
    # 중복 주소 제거 (같은 주소 두번 쓴 경우)
    address = address.split('인천광역시')[1] if address.count('인천광역시') > 1 else address
    address = address.strip()
    return address

def get_dong(address):
    try:
        clean = clean_address(address)
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": clean}, timeout=5)
        docs = res.json().get("documents", [])

        if not docs:
            return pd.Series([None, None, None])

        law_dong = docs[0]['address']['region_3depth_name']
        hang_dong = docs[0]['address']['region_3depth_h_name']
        final_dong = hang_dong if hang_dong else law_dong

        return pd.Series([law_dong, hang_dong, final_dong])

    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None, None])

df = pd.read_excel('data/행정동없는거.xlsx')
df[['법정동', '행정동', '최종동']] = df['주소'].progress_apply(get_dong)
df.to_excel('되려나.xlsx', index=False)
print('완료!')

100%|████████████████████████████████████████████████████████████████████████████████| 195/195 [00:12<00:00, 15.14it/s]

완료!


In [2]:
import requests
import pandas as pd
import re
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def clean_address(address):
    # 괄호 안 내용 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 동호수 제거
    address = re.sub(r'\s+\d+-\d+$', '', address)
    # 1~2층 제거
    address = re.sub(r'\s+\d+[~-]\d+층.*', '', address)
    # 단지 제거
    address = re.sub(r'\s+\d+단지.*', '', address)
    address = address.strip()
    return address

def get_dong(address):
    try:
        clean = clean_address(address)
        url1 = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res1 = requests.get(url1, headers=headers, params={"query": clean}, timeout=5)
        docs = res1.json().get("documents", [])

        if not docs:
            return None

        hang_dong = docs[0]['address']['region_3depth_h_name']
        law_dong = docs[0]['address']['region_3depth_name']

        # 행정동 없으면 좌표로 다시 검색
        if not hang_dong:
            x = docs[0]['x']
            y = docs[0]['y']
            url2 = "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
            res2 = requests.get(url2, headers=headers, params={"x": x, "y": y}, timeout=5)
            regions = res2.json().get("documents", [])
            for r in regions:
                if r['region_type'] == 'H':
                    hang_dong = r['region_4depth_name']
                    break

        return hang_dong if hang_dong else law_dong

    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return None

# 테스트
addresses = [
    "경기도 고양시 덕양구 세솔로 105 1704-103(삼송동,삼송마을)",
    "경기도 고양시 덕양구 백양로 126 1107-103(화정동,은빛마을)",
    "경기도 고양시 일산서구 장자길 120 685,686(덕이동)"
]

for addr in addresses:
    print(f"{addr} → {get_dong(addr)}")

경기도 고양시 덕양구 세솔로 105 1704-103(삼송동,삼송마을) → 삼송1동
경기도 고양시 덕양구 백양로 126 1107-103(화정동,은빛마을) → 화정1동
경기도 고양시 일산서구 장자길 120 685,686(덕이동) → None


In [3]:
def clean_address(address):
    # 괄호 안 내용 제거
    address = re.sub(r'\(.*?\)', '', address)
    # 동호수 제거
    address = re.sub(r'\s+\d+-\d+$', '', address)
    # 1~2층 제거
    address = re.sub(r'\s+\d+[~-]\d+층.*', '', address)
    # 단지 제거
    address = re.sub(r'\s+\d+단지.*', '', address)
    # 685,686 같은 번지수 제거
    address = re.sub(r'\s+\d+,\d+$', '', address)
    address = address.strip()
    return address

# 테스트
test = "경기도 고양시 일산서구 장자길 120 685,686(덕이동)"
print(clean_address(test))
# → 경기도 고양시 일산서구 장자길 120

경기도 고양시 일산서구 장자길 120


In [4]:
import requests
import pandas as pd
import re
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def clean_address(address):
    address = re.sub(r'\(.*?\)', '', address)
    address = re.sub(r'\s+\d+-\d+$', '', address)
    address = re.sub(r'\s+\d+[~-]\d+층.*', '', address)
    address = re.sub(r'\s+\d+단지.*', '', address)
    address = re.sub(r'\s+\d+,\d+$', '', address)
    address = address.strip()
    return address

def get_dong(address):
    try:
        clean = clean_address(address)
        url1 = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res1 = requests.get(url1, headers=headers, params={"query": clean}, timeout=5)
        docs = res1.json().get("documents", [])

        if not docs:
            return None

        hang_dong = docs[0]['address']['region_3depth_h_name']
        law_dong = docs[0]['address']['region_3depth_name']

        if not hang_dong:
            x = docs[0]['x']
            y = docs[0]['y']
            url2 = "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
            res2 = requests.get(url2, headers=headers, params={"x": x, "y": y}, timeout=5)
            regions = res2.json().get("documents", [])
            for r in regions:
                if r['region_type'] == 'H':
                    hang_dong = r['region_4depth_name']
                    break

        return hang_dong if hang_dong else law_dong

    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return None

df = pd.read_excel('되려나.xlsx')
df['행정동'] = df['주소'].progress_apply(get_dong)
df.to_excel('되려나2.xlsx', index=False)
print('완료!')

100%|████████████████████████████████████████████████████████████████████████████████| 195/195 [00:13<00:00, 14.28it/s]


완료!


In [3]:
import pandas as pd
df = pd.read_excel('data/[치안]CCTV위치.xlsx')
print(df.columns.tolist())
print(df.head(3))


['관리기관명', '소재지지번주소', '카메라대수', 'WGS84위도', 'WGS84경도']
        관리기관명           소재지지번주소  카메라대수    WGS84위도     WGS84경도
0  서울특별시 성동구청   서울특별시 행당동 19-98      3  37.558960  127.040794
1  서울특별시 성동구청  서울특별시 마장동 520-10      2  37.569137  127.037521
2  서울특별시 성동구청     서울특별시 마장동 818      3  37.570602  127.042519


In [5]:
from math import radians, sin, cos, sqrt, atan2
import requests
import pandas as pd

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

# 주소 입력창
my_address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(my_address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    print(f"내 집: {my_lat}, {my_lng}")

    # CCTV 데이터 불러오기
    df_cctv = pd.read_excel('data/[치안]CCTV위치.xlsx')

    # 거리 계산
    df_cctv['거리'] = df_cctv.apply(
        lambda row: haversine(my_lat, my_lng, row['WGS84위도'], row['WGS84경도']), axis=1
    )

    # 1km 이내 필터
    df_1km = df_cctv[df_cctv['거리'] <= 1000]

    # 결과
    count = df_1km['카메라대수'].sum()
    access_score = (1 / df_1km['거리']).sum()

    print(f"\n1km 내 CCTV 카메라 수: {count}대")
    print(f"접근성 점수: {access_score:.4f}")

주소를 입력하세요:  서울특별시 중구 퇴계로 166 2층


내 집: 37.5608939235742, 126.990166216967

1km 내 CCTV 카메라 수: 801대
접근성 점수: 0.5061


In [6]:
import streamlit as st
import pandas as pd
import requests
from math import radians, sin, cos, sqrt, atan2

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

# 사이트 제목
st.title("C-LCI 양육친화 주거지 분석")

# 주소 입력창
address = st.text_input("주소를 입력하세요")

if st.button("분석하기"):
    my_lat, my_lng = get_coord(address)

    if my_lat is None:
        st.error("주소를 찾을 수 없어요!")
    else:
        df_cctv = pd.read_excel('data/[치안]CCTV위치.xlsx')
        df_cctv['거리'] = df_cctv.apply(
            lambda row: haversine(my_lat, my_lng, row['WGS84위도'], row['WGS84경도']), axis=1
        )
        df_1km = df_cctv[df_cctv['거리'] <= 1000]

        count = df_1km['카메라대수'].sum()
        access_score = (1 / df_1km['거리']).sum()

        st.success(f"1km 내 CCTV 수: {count}대")
        st.success(f"접근성 점수: {access_score:.4f}")

2026-03-23 11:29:16.864 
  command:

    streamlit run C:\Users\asia\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-23 11:29:16.864 Session state does not function when running a script without `streamlit run`


In [7]:
import pandas as pd

df_cctv = pd.read_excel('data/[치안]CCTV위치.xlsx')
df_alarm = pd.read_excel('data/[치안]알람벨위치.xlsx')
df_police = pd.read_excel('data/[안전]경찰서위치.xlsx')
df_accident = pd.read_excel('data/[안전]교통사고빈번지역.xlsx')
df_sub = pd.read_excel('data/[안전]지구대파출소위치.xlsx')

print("CCTV:", df_cctv.columns.tolist())
print("알람벨:", df_alarm.columns.tolist())
print("경찰서:", df_police.columns.tolist())
print("교통사고:", df_accident.columns.tolist())
print("파출소:", df_sub.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'data/[치안]CCTV위치.xlsx'

In [8]:
import pandas as pd

df_cctv = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

print("CCTV:", df_cctv.columns.tolist())
print("알람벨:", df_alarm.columns.tolist())
print("경찰서:", df_police.columns.tolist())
print("교통사고:", df_accident.columns.tolist())
print("파출소:", df_sub.columns.tolist())

CCTV: ['관리기관명', '소재지지번주소', '카메라대수', 'WGS84위도', 'WGS84경도']
알람벨: ['설치위치', '소재지지번주소', '위도', '경도']
경찰서: ['경찰서명칭', '경찰서주소', '위도', '경도']
교통사고: ['시도시군구명', '지점명', '사고건수', '사상자수', '경도', '위도']
파출소: ['구분', '주소', '위도', '경도']


In [9]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

# 데이터 불러오기
df_cctv = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    print(f"내 집 좌표: {my_lat}, {my_lng}")

    # 각 접근성 점수 계산
    score_police  = get_access_score(my_lat, my_lng, df_police,  '위도', '경도', 3000)
    score_sub     = get_access_score(my_lat, my_lng, df_sub,     '위도', '경도', 3000)
    score_cctv    = get_access_score(my_lat, my_lng, df_cctv,    'WGS84위도', 'WGS84경도', 1000)
    score_alarm   = get_access_score(my_lat, my_lng, df_alarm,   '위도', '경도', 1000)
    score_accident= get_access_score(my_lat, my_lng, df_accident,'위도', '경도', 1000)

    print(f"\n경찰서 점수:   {score_police:.4f}")
    print(f"파출소 점수:   {score_sub:.4f}")
    print(f"CCTV 점수:    {score_cctv:.4f}")
    print(f"알람벨 점수:  {score_alarm:.4f}")
    print(f"교통사고 패널티: {score_accident:.4f}")

    # 안전점수 계산
    # 긍정 변수 각 0.25, 교통사고 패널티 0.05
    safety_score = (
        score_police  * 0.25 +
        score_sub     * 0.25 +
        score_cctv    * 0.25 +
        score_alarm   * 0.25
    ) - (score_accident * 0.05)

    print(f"\n안전 원시점수: {safety_score:.4f}")

주소를 입력하세요:  인천광역시 부평구 마장로 121


내 집 좌표: 37.4899042021129, 126.704778463553

경찰서 점수:   0.0004
파출소 점수:   0.0066
CCTV 점수:    0.2489
알람벨 점수:  0.2321
교통사고 패널티: 0.0000

안전 원시점수: 0.1220


In [10]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

# 치안 데이터 불러오기
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 수도권 동별 위경도 데이터 (기준점)
df_dong = pd.read_excel('data/수도권동별위경도.xlsx')  # 위도, 경도 컬럼 있는 파일

# 동별 접근성 점수 계산
def calc_safety(row):
    lat, lng = row['위도'], row['경도']
    s_police   = get_access_score(lat, lng, df_police,   '위도', '경도',       3000)
    s_sub      = get_access_score(lat, lng, df_sub,      '위도', '경도',       3000)
    s_cctv     = get_access_score(lat, lng, df_cctv,     'WGS84위도', 'WGS84경도', 1000)
    s_alarm    = get_access_score(lat, lng, df_alarm,    '위도', '경도',       1000)
    s_accident = get_access_score(lat, lng, df_accident, '위도', '경도',       1000)
    return pd.Series([s_police, s_sub, s_cctv, s_alarm, s_accident])

df_dong[['경찰서점수', '파출소점수', 'CCTV점수', '알람벨점수', '교통사고점수']] = \
    df_dong.progress_apply(calc_safety, axis=1)

# 정규화
for col in ['경찰서점수', '파출소점수', 'CCTV점수', '알람벨점수']:
    df_dong[col + '_정규화'] = (df_dong[col] - df_dong[col].min()) / \
                               (df_dong[col].max() - df_dong[col].min())

# 교통사고는 역코딩
df_dong['교통사고점수_정규화'] = 1 - (
    (df_dong['교통사고점수'] - df_dong['교통사고점수'].min()) /
    (df_dong['교통사고점수'].max() - df_dong['교통사고점수'].min())
)

# 안전점수 최종 계산
df_dong['안전점수'] = (
    df_dong['경찰서점수_정규화'] * 0.25 +
    df_dong['파출소점수_정규화'] * 0.25 +
    df_dong['CCTV점수_정규화']  * 0.25 +
    df_dong['알람벨점수_정규화'] * 0.25
) - (df_dong['교통사고점수_정규화'] * 0.05)

df_dong.to_excel('안전점수_결과.xlsx', index=False)
print('완료!')

FileNotFoundError: [Errno 2] No such file or directory: 'data/수도권동별위경도.xlsx'

In [11]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

# 데이터 불러오기
df_cctv = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    print(f"내 집 좌표: {my_lat}, {my_lng}")

    # 각 접근성 점수 계산
    score_police  = get_access_score(my_lat, my_lng, df_police,  '위도', '경도', 3000)
    score_sub     = get_access_score(my_lat, my_lng, df_sub,     '위도', '경도', 3000)
    score_cctv    = get_access_score(my_lat, my_lng, df_cctv,    'WGS84위도', 'WGS84경도', 1000)
    score_alarm   = get_access_score(my_lat, my_lng, df_alarm,   '위도', '경도', 1000)
    score_accident= get_access_score(my_lat, my_lng, df_accident,'위도', '경도', 1000)

    print(f"\n경찰서 점수:   {score_police:.4f}")
    print(f"파출소 점수:   {score_sub:.4f}")
    print(f"CCTV 점수:    {score_cctv:.4f}")
    print(f"알람벨 점수:  {score_alarm:.4f}")
    print(f"교통사고 패널티: {score_accident:.4f}")

    # 안전점수 계산
    # 긍정 변수 각 0.25, 교통사고 패널티 0.05
    safety_score = (
        score_police  * 0.25 +
        score_sub     * 0.25 +
        score_cctv    * 0.25 +
        score_alarm   * 0.25
    ) - (score_accident * 0.05)

    print(f"\n안전 원시점수: {safety_score:.4f}")

주소를 입력하세요:  서울특별시 중구 퇴계로 166 2층


내 집 좌표: 37.5608939235742, 126.990166216967

경찰서 점수:   0.0061
파출소 점수:   0.0298
CCTV 점수:    0.5061
알람벨 점수:  0.3303
교통사고 패널티: 0.0000

안전 원시점수: 0.2181


In [12]:
import pandas as pd

df = pd.read_excel('data/(동목록).xlsx')
print(df.columns.tolist())
print(df.head(3))

['동목록']
     동목록
0  청운효자동
1    사직동
2    삼청동


In [13]:
import pandas as pd

df = pd.read_excel('data/(동목록).xlsx')
print(df.columns.tolist())
print(df.head(3))

['시', '시군구', '동']
       시  시군구      동
0  서울특별시  종로구  청운효자동
1  서울특별시  종로구    사직동
2  서울특별시  종로구    삼청동


In [14]:
import pandas as pd
import requests
from tqdm import tqdm

tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_coord(row):
    address = f"{row['시']} {row['시군구']} {row['동']}"
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

df = pd.read_excel('data/(동목록).xlsx')
df['위도'], df['경도'] = zip(*df.progress_apply(get_coord, axis=1))

df.to_excel('수도권동별위경도.xlsx', index=False)
print('완료!')


100%|████████████████████████████████████████████████████████████████████████████████| 863/863 [00:51<00:00, 16.85it/s]


완료!


In [15]:
df = pd.read_excel('수도권동별위경도.xlsx')
print(df.head(5))
print(f"총 {len(df)}개 동")
print(f"None 개수: {df['위도'].isna().sum()}")


       시  시군구      동         위도          경도
0  서울특별시  종로구  청운효자동  37.584116  126.970650
1  서울특별시  종로구    사직동  37.575422  126.966320
2  서울특별시  종로구    삼청동  37.584998  126.981758
3  서울특별시  종로구    부암동  37.592414  126.964061
4  서울특별시  종로구    평창동  37.606392  126.968358
총 863개 동
None 개수: 9


In [16]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

# 데이터 불러오기
df_dong     = pd.read_excel('수도권동별위경도.xlsx').dropna(subset=['위도', '경도'])
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 동별 안전점수 계산
def calc_safety(row):
    lat, lng = row['위도'], row['경도']
    s_police   = get_access_score(lat, lng, df_police,   '위도', '경도',          3000)
    s_sub      = get_access_score(lat, lng, df_sub,      '위도', '경도',          3000)
    s_cctv     = get_access_score(lat, lng, df_cctv,     'WGS84위도', 'WGS84경도', 1000)
    s_alarm    = get_access_score(lat, lng, df_alarm,    '위도', '경도',          1000)
    s_accident = get_access_score(lat, lng, df_accident, '위도', '경도',          1000)
    return pd.Series([s_police, s_sub, s_cctv, s_alarm, s_accident])

df_dong[['경찰서점수', '파출소점수', 'CCTV점수', '알람벨점수', '교통사고점수']] = \
    df_dong.progress_apply(calc_safety, axis=1)

# 정규화
for col in ['경찰서점수', '파출소점수', 'CCTV점수', '알람벨점수']:
    df_dong[col + '_정규화'] = (df_dong[col] - df_dong[col].min()) / \
                               (df_dong[col].max() - df_dong[col].min())

# 교통사고 역코딩
df_dong['교통사고점수_정규화'] = 1 - (
    (df_dong['교통사고점수'] - df_dong['교통사고점수'].min()) /
    (df_dong['교통사고점수'].max() - df_dong['교통사고점수'].min())
)

# 안전점수 최종
df_dong['안전점수'] = (
    df_dong['경찰서점수_정규화'] * 0.25 +
    df_dong['파출소점수_정규화'] * 0.25 +
    df_dong['CCTV점수_정규화']  * 0.25 +
    df_dong['알람벨점수_정규화'] * 0.25
) - (df_dong['교통사고점수_정규화'] * 0.05)

df_dong.to_excel('안전점수_결과.xlsx', index=False)
print('완료!')

 44%|███████████████████████████████████▏                                            | 375/854 [07:29<09:34,  1.20s/it]


KeyboardInterrupt: 

In [ ]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return len(df[df['거리'] <= radius])

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

# 데이터 불러오기
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 반경 내 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          3000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          3000)
    n_cctv     = count_within(my_lat, my_lng, df_cctv,     'WGS84위도', 'WGS84경도', 1000)
    n_alarm    = count_within(my_lat, my_lng, df_alarm,    '위도', '경도',          1000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    print(f"\n경찰서 수:    {n_police}개")
    print(f"파출소 수:    {n_sub}개")
    print(f"CCTV 수:     {n_cctv}개")
    print(f"알람벨 수:   {n_alarm}개")
    print(f"교통사고 수: {n_accident}개")

    # 점수 계산
    score = (
        n_police   * 0.10 +
        n_sub      * 0.05 +
        n_cctv     * 0.01 +
        n_alarm    * 0.02
    ) - (n_accident * 0.05)

    print(f"\n안전점수: {score:.2f}")

In [1]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return len(df[df['거리'] <= radius])

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

# 데이터 불러오기
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          3000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          3000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    # 접근성 점수
    s_cctv  = get_access_score(my_lat, my_lng, df_cctv,  'WGS84위도', 'WGS84경도', 1000)
    s_alarm = get_access_score(my_lat, my_lng, df_alarm, '위도', '경도',           1000)

    print(f"\n경찰서 수:      {n_police}개")
    print(f"파출소 수:      {n_sub}개")
    print(f"CCTV 접근성:   {s_cctv:.4f}")
    print(f"알람벨 접근성: {s_alarm:.4f}")
    print(f"교통사고 수:   {n_accident}개")

    # 점수 계산
    # 경찰서 +0.1/개, 파출소 +0.05/개, CCTV/알람벨 접근성 합산, 교통사고 -0.05/개
    score = (
        n_police   * 0.10 +
        n_sub      * 0.05 +
        s_cctv          +
        s_alarm
    ) - (n_accident * 0.05)

    print(f"\n안전점수: {score:.4f}")

주소를 입력하세요:  인천광역시 부평구 마장로 121



경찰서 수:      1개
파출소 수:      9개
CCTV 접근성:   0.2489
알람벨 접근성: 0.2321
교통사고 수:   0개

안전점수: 1.0309


In [3]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return len(df[df['거리'] <= radius])

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

# 데이터 불러오기
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          3000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          2000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    # 접근성 점수
    s_cctv  = get_access_score(my_lat, my_lng, df_cctv,  'WGS84위도', 'WGS84경도', 1000)
    s_alarm = get_access_score(my_lat, my_lng, df_alarm, '위도', '경도',           1000)

    print(f"\n경찰서 수:      {n_police}개")
    print(f"파출소 수:      {n_sub}개")
    print(f"CCTV 접근성:   {s_cctv:.4f}")
    print(f"알람벨 접근성: {s_alarm:.4f}")
    print(f"교통사고 수:   {n_accident}개")

    # 점수 계산
    score = (
        n_police   * 0.10 +
        n_sub      * 0.03 +
        s_cctv          +
        s_alarm
    ) - (n_accident * 0.05)

    print(f"\n안전점수: {score:.4f}")

주소를 입력하세요:  서울특별시 중구 퇴계로 166 2층



경찰서 수:      6개
파출소 수:      18개
CCTV 접근성:   0.5061
알람벨 접근성: 0.3303
교통사고 수:   0개

안전점수: 1.9763


In [9]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return len(df[df['거리'] <= radius])

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

# 데이터 불러오기
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          1000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          1000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    # 접근성 점수
    s_cctv  = get_access_score(my_lat, my_lng, df_cctv,  'WGS84위도', 'WGS84경도', 1000)
    s_alarm = get_access_score(my_lat, my_lng, df_alarm, '위도', '경도',           1000)

    print(f"\n경찰서 수:      {n_police}개")
    print(f"파출소 수:      {n_sub}개")
    print(f"CCTV 접근성:   {s_cctv:.4f}")
    print(f"알람벨 접근성: {s_alarm:.4f}")
    print(f"교통사고 수:   {n_accident}개")

    # 점수 계산
    score = (
        n_police   * 0.10 +
        n_sub      * 0.02 +
        s_cctv          +
        s_alarm
    ) - (n_accident * 0.05)

    print(f"\n안전점수: {score:.4f}")

주소를 입력하세요:  서울특별시 중구 퇴계로 166 2층



경찰서 수:      1개
파출소 수:      5개
CCTV 접근성:   0.5061
알람벨 접근성: 0.3303
교통사고 수:   0개

안전점수: 1.0363


In [17]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return len(df[df['거리'] <= radius])

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

# 데이터 불러오기
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          1000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          1000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    # 접근성 점수
    s_cctv  = get_access_score(my_lat, my_lng, df_cctv,  'WGS84위도', 'WGS84경도', 1000)
    s_alarm = get_access_score(my_lat, my_lng, df_alarm, '위도', '경도',           1000)

    # 점수 계산
    score_police   = n_police   * 0.10
    score_sub      = n_sub      * 0.03
    score_accident = n_accident * 0.05

    score = (
        score_police +
        score_sub    +
        s_cctv       +
        s_alarm
    ) - score_accident

    print(f"\n경찰서 점수:      {score_police:.4f}")
    print(f"파출소 점수:      {score_sub:.4f}")
    print(f"CCTV 접근성:     {s_cctv:.4f}")
    print(f"알람벨 접근성:   {s_alarm:.4f}")
    print(f"교통사고 패널티: -{score_accident:.4f}")
    print(f"\n안전점수: {score:.4f}")

주소를 입력하세요:  서울특별시 마포구 합정동 366-1 



경찰서 점수:      0.0000
파출소 점수:      0.0300
CCTV 접근성:     0.5728
알람벨 접근성:   0.4906
교통사고 패널티: -0.0000

안전점수: 1.0934


In [20]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# ============================================================
# 정규화 기준값
# ============================================================
POLICE_MAX  = 2    # 1km 내 경찰서 2개 이상 최고점
SUB_MAX     = 5    # 1km 내 파출소 5개 이상 최고점
CCTV_MAX    = 0.8  # CCTV 접근성 0.8 이상 최고점
ALARM_MAX   = 0.8  # 알람벨 접근성 0.8 이상 최고점
ACC_MAX     = 2    # 1km 내 교통사고 2개 이상 최고점(패널티)

# ============================================================
# 데이터 로드
# ============================================================
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          1000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          1000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    # 접근성 점수
    s_cctv  = get_access_score(my_lat, my_lng, df_cctv,  'WGS84위도', 'WGS84경도', 1000)
    s_alarm = get_access_score(my_lat, my_lng, df_alarm, '위도', '경도',           1000)

    # 정규화
    v1 = normalize(n_police,   0, POLICE_MAX)               # 경찰서
    v2 = normalize(n_sub,      0, SUB_MAX)                  # 파출소
    v3 = normalize(s_cctv,     0, CCTV_MAX)                 # CCTV
    v4 = normalize(s_alarm,    0, ALARM_MAX)                # 알람벨
    v5 = normalize(n_accident, 0, ACC_MAX, reverse=True)    # 교통사고 역방향

    # 점수 계산 (교통사고 0.1 패널티)
    score = round((v1 + v2 + v3 + v4) / 4 * 100 - v5 * 0.1 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"경찰서 (1km):      {n_police}개  → V1: {v1:.4f}")
    print(f"파출소 (1km):      {n_sub}개  → V2: {v2:.4f}")
    print(f"CCTV 접근성:      {s_cctv:.4f}  → V3: {v3:.4f}")
    print(f"알람벨 접근성:    {s_alarm:.4f}  → V4: {v4:.4f}")
    print(f"교통사고 (1km):   {n_accident}개  → V5: {v5:.4f} (패널티)")
    print()
    print(f"안전점수 = (V1+V2+V3+V4)/4 × 100 - V5 × 10")
    print(f"         = {score}점")

주소를 입력하세요:  인천광역시 부평구 마장로 121



📍 인천광역시 부평구 마장로 121
   좌표: (37.4899042021129, 126.704778463553)

경찰서 (1km):      0개  → V1: 0.0000
파출소 (1km):      1개  → V2: 0.2000
CCTV 접근성:      0.2489  → V3: 0.3111
알람벨 접근성:    0.2321  → V4: 0.2901
교통사고 (1km):   0개  → V5: 1.0000 (패널티)

안전점수 = (V1+V2+V3+V4)/4 × 100 - V5 × 10
         = 10.03점


In [22]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# ============================================================
# 정규화 기준값
# ============================================================
POLICE_MAX  = 2    # 3km 내 경찰서 2개 이상 최고점
SUB_MAX     = 7    # 2km 내 파출소 7개 이상 최고점
CCTV_MAX    = 0.8  # CCTV 접근성 0.8 이상 최고점
ALARM_MAX   = 0.8  # 알람벨 접근성 0.8 이상 최고점
ACC_MAX     = 2    # 1km 내 교통사고 2개 이상 최고점(패널티)

# ============================================================
# 데이터 로드
# ============================================================
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          3000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          2000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    # 접근성 점수
    s_cctv  = get_access_score(my_lat, my_lng, df_cctv,  'WGS84위도', 'WGS84경도', 1000)
    s_alarm = get_access_score(my_lat, my_lng, df_alarm, '위도', '경도',           1000)

    # 정규화
    v1 = normalize(n_police, 0, POLICE_MAX)
    v2 = normalize(n_sub,    0, SUB_MAX)
    v3 = normalize(s_cctv,   0, CCTV_MAX)
    v4 = normalize(s_alarm,  0, ALARM_MAX)

    # 교통사고 패널티 (0이면 패널티 없음)
    penalty = 0
    if n_accident > 0:
        v5 = normalize(n_accident, 0, ACC_MAX, reverse=True)
        penalty = round(v5 * 0.1 * 100, 2)
    else:
        v5 = 0.0

    # 점수 계산
    score = round((v1 + v2 + v3 + v4) / 4 * 100 - penalty, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"경찰서 (3km):      {n_police}개  → V1: {v1:.4f}")
    print(f"파출소 (2km):      {n_sub}개  → V2: {v2:.4f}")
    print(f"CCTV 접근성:      {s_cctv:.4f}  → V3: {v3:.4f}")
    print(f"알람벨 접근성:    {s_alarm:.4f}  → V4: {v4:.4f}")
    print(f"교통사고 (1km):   {n_accident}개  → V5: {v5:.4f} (패널티: -{penalty})")
    print()
    print(f"안전점수 = (V1+V2+V3+V4)/4 × 100 - 패널티")
    print(f"         = {score}점")

주소를 입력하세요:  서울특별시 중구 퇴계로 166 2층



📍 서울특별시 중구 퇴계로 166 2층
   좌표: (37.5608939235742, 126.990166216967)

경찰서 (3km):      6개  → V1: 1.0000
파출소 (2km):      18개  → V2: 1.0000
CCTV 접근성:      0.5061  → V3: 0.6326
알람벨 접근성:    0.3303  → V4: 0.4128
교통사고 (1km):   0개  → V5: 0.0000 (패널티: -0)

안전점수 = (V1+V2+V3+V4)/4 × 100 - 패널티
         = 76.13점


In [24]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# ============================================================
# 정규화 기준값
# ============================================================
POLICE_MAX = 2    # 3km 내 경찰서 2개 이상 최고점
SUB_MAX    = 7    # 2km 내 파출소 7개 이상 최고점
CCTV_MAX   = 0.8  # CCTV 접근성 0.8 이상 최고점
ALARM_MAX  = 0.8  # 알람벨 접근성 0.8 이상 최고점

# ============================================================
# 데이터 로드
# ============================================================
df_cctv     = pd.read_excel('data/(치안)CCTV위치.xlsx')
df_alarm    = pd.read_excel('data/(치안)알람벨위치.xlsx')
df_police   = pd.read_excel('data/(안전)경찰서위치.xlsx')
df_accident = pd.read_excel('data/(안전)교통사고빈번지역.xlsx')
df_sub      = pd.read_excel('data/(안전)지구대파출소위치.xlsx')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 개수 계산
    n_police   = count_within(my_lat, my_lng, df_police,   '위도', '경도',          3000)
    n_sub      = count_within(my_lat, my_lng, df_sub,      '위도', '경도',          2000)
    n_accident = count_within(my_lat, my_lng, df_accident, '위도', '경도',          1000)

    # 접근성 점수
    s_cctv  = get_access_score(my_lat, my_lng, df_cctv,  'WGS84위도', 'WGS84경도', 1000)
    s_alarm = get_access_score(my_lat, my_lng, df_alarm, '위도', '경도',           1000)

    # 정규화
    v1 = normalize(n_police, 0, POLICE_MAX)
    v2 = normalize(n_sub,    0, SUB_MAX)
    v3 = normalize(s_cctv,   0, CCTV_MAX)
    v4 = normalize(s_alarm,  0, ALARM_MAX)

    # 교통사고 패널티 (1개당 3점 고정)
    penalty = n_accident * 3

    # 점수 계산
    score = round((v1 + v2 + v3 + v4) / 4 * 100 - penalty, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"경찰서 (3km):      {n_police}개  → V1: {v1:.4f}")
    print(f"파출소 (2km):      {n_sub}개  → V2: {v2:.4f}")
    print(f"CCTV 접근성:      {s_cctv:.4f}  → V3: {v3:.4f}")
    print(f"알람벨 접근성:    {s_alarm:.4f}  → V4: {v4:.4f}")
    print(f"교통사고 (1km):   {n_accident}개  → 패널티: -{penalty}점")
    print()
    print(f"안전점수 = (V1+V2+V3+V4)/4 × 100 - 패널티")
    print(f"         = {score}점")

주소를 입력하세요:  서울특별시 용산구 대사관로 59



📍 서울특별시 용산구 대사관로 59
   좌표: (37.5342808357653, 127.003878033451)

경찰서 (3km):      0개  → V1: 0.0000
파출소 (2km):      4개  → V2: 0.5714
CCTV 접근성:      0.8335  → V3: 1.0000
알람벨 접근성:    0.5750  → V4: 0.7187
교통사고 (1km):   1개  → 패널티: -3점

안전점수 = (V1+V2+V3+V4)/4 × 100 - 패널티
         = 54.25점


In [25]:
import pandas as pd

df_hospital = pd.read_excel('data/(의료)주변소아과.xlsx')
df_welfare = pd.read_excel('data/(복지)통합아동지원센터.xlsx')

print("소아과:", df_hospital.columns.tolist())
print("아동센터:", df_welfare.columns.tolist())

소아과: ['Unnamed: 0', '요양기관명', '진료과목코드명', '과목별 전문의수', '선택진료 의사수', '종별코드명', '시군구코드명', '읍면동', '주소', '좌표(X)', '좌표(Y)', '행정동']
아동센터: ['Unnamed: 0', '시설명', '주소', 'Y좌표값', 'X좌표값', '행정동', '법정동']


In [26]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# ============================================================
# 정규화 기준값
# ============================================================
HOSPITAL_MAX = 0.8  # 소아과 접근성 0.8 이상 최고점
WELFARE_MAX  = 3    # 2km 내 아동센터 3개 이상 최고점

# ============================================================
# 데이터 로드
# ============================================================
df_hospital = pd.read_excel('data/(의료)주변소아과.xlsx')
df_welfare  = pd.read_excel('data/(복지)통합아동지원센터.xlsx')

# ============================================================
# 주소 입력
# ============================================================
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 소아과 접근성 점수 (Y좌표=위도, X좌표=경도)
    s_hospital = get_access_score(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)

    # 아동센터 개수 (Y좌표값=위도, X좌표값=경도)
    n_welfare  = count_within(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)

    # 정규화
    v1 = normalize(s_hospital, 0, HOSPITAL_MAX)
    v2 = normalize(n_welfare,  0, WELFARE_MAX)

print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"소아과 접근성 (2km):      {s_hospital:.4f}  → V1: {v1:.4f}")
    print(f"통합아동지원센터 (2km):   {n_welfare}개  → V2: {v2:.4f}")
    print()
    print(f"의료·복지 점수 = (V1+V2)/2 × 100")
    print(f"              = {score}점")

IndentationError: unexpected indent (3141056236.py, line 78)

In [28]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# 정규화 기준값
HOSPITAL_MAX = 0.8
WELFARE_MAX  = 3

# 데이터 로드
df_hospital = pd.read_excel('data/(의료)주변소아과.xlsx')
df_welfare  = pd.read_excel('data/(복지)통합아동지원센터.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 소아과
    s_hospital = get_access_score(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)
    n_hospital = count_within(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)

    # 아동센터
    n_welfare = count_within(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)

    # 정규화
    v1 = normalize(s_hospital, 0, HOSPITAL_MAX)
    v2 = normalize(n_welfare,  0, WELFARE_MAX)

    # 점수 계산
    score = round((v1 + v2) / 2 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"소아과 개수 (2km):             {n_hospital}개")
    print(f"소아과 접근성 (2km):           {s_hospital:.4f}  → V1: {v1:.4f}")
    print(f"통합아동지원센터 (2km):        {n_welfare}개  → V2: {v2:.4f}")
    print()
    print(f"의료·복지 점수 = (V1+V2)/2 × 100")
    print(f"              = {score}점")

주소를 입력하세요:  서울특별시 중구 퇴계로 166 2층



📍 서울특별시 중구 퇴계로 166 2층
   좌표: (37.5608939235742, 126.990166216967)

소아과 개수 (2km):             19개
소아과 접근성 (2km):           0.0118  → V1: 0.0148
통합아동지원센터 (2km):        5개  → V2: 1.0000

의료·복지 점수 = (V1+V2)/2 × 100
              = 50.74점


In [29]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# 정규화 기준값
HOSPITAL_MAX = 0.02  # 소아과 접근성 최고점
WELFARE_MAX  = 0.01  # 아동센터 접근성 최고점

# 데이터 로드
df_hospital = pd.read_excel('data/(의료)주변소아과.xlsx')
df_welfare  = pd.read_excel('data/(복지)통합아동지원센터.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 소아과
    s_hospital = get_access_score(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)
    n_hospital = count_within(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)

    # 아동센터
    s_welfare = get_access_score(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)
    n_welfare = count_within(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)

    # 정규화
    v1 = normalize(s_hospital, 0, HOSPITAL_MAX)
    v2 = normalize(s_welfare,  0, WELFARE_MAX)

    # 점수 계산
    score = round((v1 + v2) / 2 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"소아과 개수 (2km):             {n_hospital}개")
    print(f"소아과 접근성 (2km):           {s_hospital:.4f}  → V1: {v1:.4f}")
    print(f"통합아동지원센터 개수 (2km):   {n_welfare}개")
    print(f"통합아동지원센터 접근성 (2km): {s_welfare:.4f}  → V2: {v2:.4f}")
    print()
    print(f"의료·복지 점수 = (V1+V2)/2 × 100")
    print(f"              = {score}점")

주소를 입력하세요:  인천광역시 부평구 마장로 121



📍 인천광역시 부평구 마장로 121
   좌표: (37.4899042021129, 126.704778463553)

소아과 개수 (2km):             89개
소아과 접근성 (2km):           0.0718  → V1: 1.0000
통합아동지원센터 개수 (2km):   12개
통합아동지원센터 접근성 (2km): 0.0104  → V2: 1.0000

의료·복지 점수 = (V1+V2)/2 × 100
              = 100.0점


In [47]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# 정규화 기준값
HOSPITAL_MAX = 0.03  # 소아과 접근성 최고점
WELFARE_MAX  = 0.01  # 아동센터 접근성 최고점

# 데이터 로드
df_hospital = pd.read_excel('data/(의료)주변소아과.xlsx')
df_welfare  = pd.read_excel('data/(복지)통합아동지원센터.xlsx')

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
    # 소아과 1km
    s_hospital = get_access_score(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)
    n_hospital = count_within(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)

    # 아동센터 1km
    s_welfare = get_access_score(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)
    n_welfare = count_within(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)

    # 정규화
    v1 = normalize(s_hospital, 0, HOSPITAL_MAX)
    v2 = normalize(s_welfare,  0, WELFARE_MAX)

    # 점수 계산
    score = round((v1 + v2) / 2 * 100, 2)

    # 결과 출력
    print(f"\n📍 {address}")
    print(f"   좌표: ({my_lat}, {my_lng})")
    print()
    print(f"소아과 개수 (2km):             {n_hospital}개")
    print(f"소아과 접근성 (2km):           {s_hospital:.4f}  → V1: {v1:.4f}")
    print(f"통합아동지원센터 개수 (2km):   {n_welfare}개")
    print(f"통합아동지원센터 접근성 (2km): {s_welfare:.4f}  → V2: {v2:.4f}")
    print()
    print(f"의료·복지 점수 = (V1+V2)/2 × 100")
    print(f"              = {score}점")

주소를 입력하세요:  인천광역시 부평구 마장로 121



📍 인천광역시 부평구 마장로 121
   좌표: (37.4899042021129, 126.704778463553)

소아과 개수 (2km):             23개
소아과 접근성 (2km):           0.0163  → V1: 0.5442
통합아동지원센터 개수 (2km):   12개
통합아동지원센터 접근성 (2km): 0.0104  → V2: 1.0000

의료·복지 점수 = (V1+V2)/2 × 100
              = 77.21점


In [1]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [2]:
streamlit hello

SyntaxError: invalid syntax (2773187961.py, line 1)

In [6]:
import pandas as pd
df =pd.read_excel('data/(동목록).xlsx')
df

,시,시군구,동
0,서울특별시,종로구,청운효자동
1,서울특별시,종로구,사직동
2,서울특별시,종로구,삼청동
3,서울특별시,종로구,부암동
4,서울특별시,종로구,평창동
...,...,...,...
858,경기도,김포시,풍무동
859,경기도,김포시,장기동
860,경기도,김포시,구래동
861,경기도,김포시,마산동


In [7]:
df['동'].str.replace('제(\d)', r'\1', regex=True)
df

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\asia\AppData\Local\Temp\ipykernel_21004\3940962430.py:1: SyntaxWarning: invalid escape sequence '\d'
  df['동'].str.replace('제(\d)', r'\1', regex=True)


,시,시군구,동
0,서울특별시,종로구,청운효자동
1,서울특별시,종로구,사직동
2,서울특별시,종로구,삼청동
3,서울특별시,종로구,부암동
4,서울특별시,종로구,평창동
...,...,...,...
858,경기도,김포시,풍무동
859,경기도,김포시,장기동
860,경기도,김포시,구래동
861,경기도,김포시,마산동


In [8]:
df['동'] = df['동'].str.replace(r'제(\d)', r'\1', regex=True)
df

,시,시군구,동
0,서울특별시,종로구,청운효자동
1,서울특별시,종로구,사직동
2,서울특별시,종로구,삼청동
3,서울특별시,종로구,부암동
4,서울특별시,종로구,평창동
...,...,...,...
858,경기도,김포시,풍무동
859,경기도,김포시,장기동
860,경기도,김포시,구래동
861,경기도,김포시,마산동


In [10]:
df.to_excel('동목록제거.xlsx', index=False)

In [1]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
res = requests.get(url, headers=headers, params={"query": "서울특별시 동대문구 전농동 127-359"})
docs = res.json().get("documents", [])
print(docs[0]['y'], docs[0]['x'])

37.5805784785406 127.050917273005


In [3]:
import pandas as pd
df = pd.read_excel("data/서울아파트매매.xlsx")
df

,NO,시군구,번지,본번,부번,단지명,거래금액(만원),도로명,주택유형,합친거
0,1,서울특별시 서대문구 남가좌동,385,385,0,DMC파크뷰자이1단지,"215,000",가재울미래로 2,아파트,서울특별시 서대문구 남가좌동 385
1,2,서울특별시 관악구 봉천동,1724,1724,0,관악월드메르디앙,"79,000",관악로28길 80,아파트,서울특별시 관악구 봉천동 1724
2,3,서울특별시 관악구 봉천동,1707-1,1707,1,은천2단지아파트,"69,000",인헌12길 46-1,아파트,서울특별시 관악구 봉천동 1707-1
3,4,서울특별시 관악구 신림동,1727,1727,0,그대家,"53,500",난곡로 167,아파트,서울특별시 관악구 신림동 1727
4,5,서울특별시 용산구 한남동,829,829,0,나인원한남,"1,565,000",한남대로 91,아파트,서울특별시 용산구 한남동 829
...,...,...,...,...,...,...,...,...,...,...
76821,76822,서울특별시 강서구 화곡동,1074-18,1074,18,서부인터빌아파트,"59,500",화곡로26가길 10,아파트,서울특별시 강서구 화곡동 1074-18
76822,76823,서울특별시 강서구 화곡동,1087,1087,0,화곡1차보람더하임,"40,000",화곡로14길 19,아파트,서울특별시 강서구 화곡동 1087
76823,76824,서울특별시 양천구 신정동,311,311,0,목동신시가지10,"135,800",목동서로 400,아파트,서울특별시 양천구 신정동 311
76824,76825,서울특별시 양천구 신정동,311,311,0,목동신시가지10,"132,000",목동서로 400,아파트,서울특별시 양천구 신정동 311


In [12]:
df['거래금액_평균'] = df.groupby('합친거')['거래금액(만원)'].transform('mean')


In [15]:
import pandas as pd
df = pd.read_excel("data/서울아파트매매.xlsx")
df

,NO,시군구,번지,본번,부번,단지명,거래금액(만원),도로명,주택유형,합친거
0,1,서울특별시 서대문구 남가좌동,385,385,0,DMC파크뷰자이1단지,"215,000",가재울미래로 2,아파트,서울특별시 서대문구 남가좌동 385
1,2,서울특별시 관악구 봉천동,1724,1724,0,관악월드메르디앙,"79,000",관악로28길 80,아파트,서울특별시 관악구 봉천동 1724
2,3,서울특별시 관악구 봉천동,1707-1,1707,1,은천2단지아파트,"69,000",인헌12길 46-1,아파트,서울특별시 관악구 봉천동 1707-1
3,4,서울특별시 관악구 신림동,1727,1727,0,그대家,"53,500",난곡로 167,아파트,서울특별시 관악구 신림동 1727
4,5,서울특별시 용산구 한남동,829,829,0,나인원한남,"1,565,000",한남대로 91,아파트,서울특별시 용산구 한남동 829
...,...,...,...,...,...,...,...,...,...,...
76821,76822,서울특별시 강서구 화곡동,1074-18,1074,18,서부인터빌아파트,"59,500",화곡로26가길 10,아파트,서울특별시 강서구 화곡동 1074-18
76822,76823,서울특별시 강서구 화곡동,1087,1087,0,화곡1차보람더하임,"40,000",화곡로14길 19,아파트,서울특별시 강서구 화곡동 1087
76823,76824,서울특별시 양천구 신정동,311,311,0,목동신시가지10,"135,800",목동서로 400,아파트,서울특별시 양천구 신정동 311
76824,76825,서울특별시 양천구 신정동,311,311,0,목동신시가지10,"132,000",목동서로 400,아파트,서울특별시 양천구 신정동 311


In [18]:
df['거래금액(만원)'] = df['거래금액(만원)'].astype(str).str.replace(',', '').str.strip()
df['거래금액(만원)'] = pd.to_numeric(df['거래금액(만원)'], errors='coerce')

# 2. 그다음 평균 추가

df['거래금액_평균'] = df.groupby('합친거')['거래금액(만원)'].transform('mean')
df

,NO,시군구,번지,본번,부번,단지명,거래금액(만원),도로명,주택유형,합친거,거래금액_평균
0,1,서울특별시 서대문구 남가좌동,385,385,0,DMC파크뷰자이1단지,215000,가재울미래로 2,아파트,서울특별시 서대문구 남가좌동 385,1.378842e+05
1,2,서울특별시 관악구 봉천동,1724,1724,0,관악월드메르디앙,79000,관악로28길 80,아파트,서울특별시 관악구 봉천동 1724,7.360000e+04
2,3,서울특별시 관악구 봉천동,1707-1,1707,1,은천2단지아파트,69000,인헌12길 46-1,아파트,서울특별시 관악구 봉천동 1707-1,6.039167e+04
3,4,서울특별시 관악구 신림동,1727,1727,0,그대家,53500,난곡로 167,아파트,서울특별시 관악구 신림동 1727,5.728077e+04
4,5,서울특별시 용산구 한남동,829,829,0,나인원한남,1565000,한남대로 91,아파트,서울특별시 용산구 한남동 829,1.469833e+06
...,...,...,...,...,...,...,...,...,...,...,...
76821,76822,서울특별시 강서구 화곡동,1074-18,1074,18,서부인터빌아파트,59500,화곡로26가길 10,아파트,서울특별시 강서구 화곡동 1074-18,5.960000e+04
76822,76823,서울특별시 강서구 화곡동,1087,1087,0,화곡1차보람더하임,40000,화곡로14길 19,아파트,서울특별시 강서구 화곡동 1087,3.575000e+04
76823,76824,서울특별시 양천구 신정동,311,311,0,목동신시가지10,135800,목동서로 400,아파트,서울특별시 양천구 신정동 311,2.030055e+05
76824,76825,서울특별시 양천구 신정동,311,311,0,목동신시가지10,132000,목동서로 400,아파트,서울특별시 양천구 신정동 311,2.030055e+05


In [19]:
df.to_excel("서울아파트매매가.xlsx")
print("완료")

완료


In [20]:
import pandas as pd
df = pd.read_excel("data/서울아파트매매가.xlsx")

df['거래금액_평균'] = (df['거래금액_평균'] * 10000).astype(int)
df

,Unnamed: 0,NO,시군구,번지,본번,부번,단지명,거래금액(만원),도로명,주택유형,합친거,거래금액_평균
0,0,1,서울특별시 서대문구 남가좌동,385,385,0,DMC파크뷰자이1단지,215000,가재울미래로 2,아파트,서울특별시 서대문구 남가좌동 385,1378842430
1,1,2,서울특별시 관악구 봉천동,1724,1724,0,관악월드메르디앙,79000,관악로28길 80,아파트,서울특별시 관악구 봉천동 1724,736000000
2,2,3,서울특별시 관악구 봉천동,1707-1,1707,1,은천2단지아파트,69000,인헌12길 46-1,아파트,서울특별시 관악구 봉천동 1707-1,603916666
3,3,4,서울특별시 관악구 신림동,1727,1727,0,그대家,53500,난곡로 167,아파트,서울특별시 관악구 신림동 1727,572807692
4,4,5,서울특별시 용산구 한남동,829,829,0,나인원한남,1565000,한남대로 91,아파트,서울특별시 용산구 한남동 829,-2147483648
...,...,...,...,...,...,...,...,...,...,...,...,...
76821,76821,76822,서울특별시 강서구 화곡동,1074-18,1074,18,서부인터빌아파트,59500,화곡로26가길 10,아파트,서울특별시 강서구 화곡동 1074-18,596000000
76822,76822,76823,서울특별시 강서구 화곡동,1087,1087,0,화곡1차보람더하임,40000,화곡로14길 19,아파트,서울특별시 강서구 화곡동 1087,357500000
76823,76823,76824,서울특별시 양천구 신정동,311,311,0,목동신시가지10,135800,목동서로 400,아파트,서울특별시 양천구 신정동 311,2030054545
76824,76824,76825,서울특별시 양천구 신정동,311,311,0,목동신시가지10,132000,목동서로 400,아파트,서울특별시 양천구 신정동 311,2030054545


In [22]:
import pandas as pd
df = pd.read_excel("data/서울아파트매매가.xlsx")

df['거래금액_평균'] = (df['거래금액_평균']).astype(int)
df

,Unnamed: 0,NO,시군구,번지,본번,부번,단지명,거래금액(만원),도로명,주택유형,합친거,거래금액_평균
0,0,1,서울특별시 서대문구 남가좌동,385,385,0,DMC파크뷰자이1단지,215000,가재울미래로 2,아파트,서울특별시 서대문구 남가좌동 385,137884
1,1,2,서울특별시 관악구 봉천동,1724,1724,0,관악월드메르디앙,79000,관악로28길 80,아파트,서울특별시 관악구 봉천동 1724,73600
2,2,3,서울특별시 관악구 봉천동,1707-1,1707,1,은천2단지아파트,69000,인헌12길 46-1,아파트,서울특별시 관악구 봉천동 1707-1,60391
3,3,4,서울특별시 관악구 신림동,1727,1727,0,그대家,53500,난곡로 167,아파트,서울특별시 관악구 신림동 1727,57280
4,4,5,서울특별시 용산구 한남동,829,829,0,나인원한남,1565000,한남대로 91,아파트,서울특별시 용산구 한남동 829,1469833
...,...,...,...,...,...,...,...,...,...,...,...,...
76821,76821,76822,서울특별시 강서구 화곡동,1074-18,1074,18,서부인터빌아파트,59500,화곡로26가길 10,아파트,서울특별시 강서구 화곡동 1074-18,59600
76822,76822,76823,서울특별시 강서구 화곡동,1087,1087,0,화곡1차보람더하임,40000,화곡로14길 19,아파트,서울특별시 강서구 화곡동 1087,35750
76823,76823,76824,서울특별시 양천구 신정동,311,311,0,목동신시가지10,135800,목동서로 400,아파트,서울특별시 양천구 신정동 311,203005
76824,76824,76825,서울특별시 양천구 신정동,311,311,0,목동신시가지10,132000,목동서로 400,아파트,서울특별시 양천구 신정동 311,203005


In [24]:
df.to_excel("서울아파트매매가최종.xlsx")

In [26]:
import pandas as pd
df = pd.read_excel("data/연립다세대(매매)_실거래가.xlsx")
df

,시군구,번지,본번,부번,건물명,거래금액(만원),도로명,주택유형
0,서울특별시 성북구 삼선동5가,90-1,90,1,타임아트빌(90-1),"27,000",삼선교로18나길 2,다세대
1,서울특별시 중랑구 묵동,240-146,240,146,한신그린빌,"28,000",동일로157길 52-10,다세대
2,서울특별시 은평구 응암동,676-20,676,20,제니더플래인엣명지,"25,000",백련산로 9,다세대
3,서울특별시 은평구 불광동,105-50,105,50,"불광아이빌3(A,B)","20,000",연서로40길 12-7,다세대
4,서울특별시 강서구 화곡동,56-481,56,481,더스위트,"19,000",초록마을로16길 21,다세대
...,...,...,...,...,...,...,...,...
36605,서울특별시 광진구 자양동,617-39,617,39,그레이스빌,"28,000",자양로13다길 8,다세대
36606,서울특별시 동대문구 회기동,103-262,103,262,노블레스,"44,700",회기로10가길 35-6,다세대
36607,서울특별시 중랑구 상봉동,103-4,103,4,라엘103,"32,000",봉우재로33길 55-6,다세대
36608,서울특별시 광진구 구의동,61-38,61,38,용빌라,"30,000",자양로38길 55,다세대


In [28]:
df['거래금액(만원)'] = df['거래금액(만원)'].astype(str).str.replace(',', '').str.strip()
df['거래금액(만원)'] = pd.to_numeric(df['거래금액(만원)'], errors='coerce')

# 2. 그다음 평균 추가

df['거래금액_평균'] = df.groupby('합친거')['거래금액(만원)'].transform('mean')
df

KeyError: '합친거'

In [29]:
import pandas as pd
df = pd.read_excel("data/연립다세대(매매)_실거래가.xlsx")
df

,시군구,번지,본번,부번,건물명,거래금액(만원),도로명,주택유형,합친거
0,서울특별시 성북구 삼선동5가,90-1,90,1,타임아트빌(90-1),"27,000",삼선교로18나길 2,다세대,서울특별시 성북구 삼선동5가 90-1
1,서울특별시 중랑구 묵동,240-146,240,146,한신그린빌,"28,000",동일로157길 52-10,다세대,서울특별시 중랑구 묵동 240-146
2,서울특별시 은평구 응암동,676-20,676,20,제니더플래인엣명지,"25,000",백련산로 9,다세대,서울특별시 은평구 응암동 676-20
3,서울특별시 은평구 불광동,105-50,105,50,"불광아이빌3(A,B)","20,000",연서로40길 12-7,다세대,서울특별시 은평구 불광동 105-50
4,서울특별시 강서구 화곡동,56-481,56,481,더스위트,"19,000",초록마을로16길 21,다세대,서울특별시 강서구 화곡동 56-481
...,...,...,...,...,...,...,...,...,...
36605,서울특별시 광진구 자양동,617-39,617,39,그레이스빌,"28,000",자양로13다길 8,다세대,서울특별시 광진구 자양동 617-39
36606,서울특별시 동대문구 회기동,103-262,103,262,노블레스,"44,700",회기로10가길 35-6,다세대,서울특별시 동대문구 회기동 103-262
36607,서울특별시 중랑구 상봉동,103-4,103,4,라엘103,"32,000",봉우재로33길 55-6,다세대,서울특별시 중랑구 상봉동 103-4
36608,서울특별시 광진구 구의동,61-38,61,38,용빌라,"30,000",자양로38길 55,다세대,서울특별시 광진구 구의동 61-38


In [30]:
df['거래금액(만원)'] = df['거래금액(만원)'].astype(str).str.replace(',', '').str.strip()
df['거래금액(만원)'] = pd.to_numeric(df['거래금액(만원)'], errors='coerce')

# 2. 그다음 평균 추가

df['거래금액_평균'] = df.groupby('합친거')['거래금액(만원)'].transform('mean')
df

,시군구,번지,본번,부번,건물명,거래금액(만원),도로명,주택유형,합친거,거래금액_평균
0,서울특별시 성북구 삼선동5가,90-1,90,1,타임아트빌(90-1),27000,삼선교로18나길 2,다세대,서울특별시 성북구 삼선동5가 90-1,39700.000000
1,서울특별시 중랑구 묵동,240-146,240,146,한신그린빌,28000,동일로157길 52-10,다세대,서울특별시 중랑구 묵동 240-146,33500.000000
2,서울특별시 은평구 응암동,676-20,676,20,제니더플래인엣명지,25000,백련산로 9,다세대,서울특별시 은평구 응암동 676-20,25000.000000
3,서울특별시 은평구 불광동,105-50,105,50,"불광아이빌3(A,B)",20000,연서로40길 12-7,다세대,서울특별시 은평구 불광동 105-50,24125.000000
4,서울특별시 강서구 화곡동,56-481,56,481,더스위트,19000,초록마을로16길 21,다세대,서울특별시 강서구 화곡동 56-481,19000.000000
...,...,...,...,...,...,...,...,...,...,...
36605,서울특별시 광진구 자양동,617-39,617,39,그레이스빌,28000,자양로13다길 8,다세대,서울특별시 광진구 자양동 617-39,27333.333333
36606,서울특별시 동대문구 회기동,103-262,103,262,노블레스,44700,회기로10가길 35-6,다세대,서울특별시 동대문구 회기동 103-262,45450.000000
36607,서울특별시 중랑구 상봉동,103-4,103,4,라엘103,32000,봉우재로33길 55-6,다세대,서울특별시 중랑구 상봉동 103-4,30857.142857
36608,서울특별시 광진구 구의동,61-38,61,38,용빌라,30000,자양로38길 55,다세대,서울특별시 광진구 구의동 61-38,30000.000000


In [31]:
df['거래금액_평균'] = (df['거래금액_평균']).astype(int)
df

,시군구,번지,본번,부번,건물명,거래금액(만원),도로명,주택유형,합친거,거래금액_평균
0,서울특별시 성북구 삼선동5가,90-1,90,1,타임아트빌(90-1),27000,삼선교로18나길 2,다세대,서울특별시 성북구 삼선동5가 90-1,39700
1,서울특별시 중랑구 묵동,240-146,240,146,한신그린빌,28000,동일로157길 52-10,다세대,서울특별시 중랑구 묵동 240-146,33500
2,서울특별시 은평구 응암동,676-20,676,20,제니더플래인엣명지,25000,백련산로 9,다세대,서울특별시 은평구 응암동 676-20,25000
3,서울특별시 은평구 불광동,105-50,105,50,"불광아이빌3(A,B)",20000,연서로40길 12-7,다세대,서울특별시 은평구 불광동 105-50,24125
4,서울특별시 강서구 화곡동,56-481,56,481,더스위트,19000,초록마을로16길 21,다세대,서울특별시 강서구 화곡동 56-481,19000
...,...,...,...,...,...,...,...,...,...,...
36605,서울특별시 광진구 자양동,617-39,617,39,그레이스빌,28000,자양로13다길 8,다세대,서울특별시 광진구 자양동 617-39,27333
36606,서울특별시 동대문구 회기동,103-262,103,262,노블레스,44700,회기로10가길 35-6,다세대,서울특별시 동대문구 회기동 103-262,45450
36607,서울특별시 중랑구 상봉동,103-4,103,4,라엘103,32000,봉우재로33길 55-6,다세대,서울특별시 중랑구 상봉동 103-4,30857
36608,서울특별시 광진구 구의동,61-38,61,38,용빌라,30000,자양로38길 55,다세대,서울특별시 광진구 구의동 61-38,30000


In [32]:
df.to_excel("서울연립다세대매매가.xlsx")
print("완료")

완료


In [34]:
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
res = requests.get(url, headers=headers, params={"query": "모래내로"})
docs = res.json().get("documents", [])
print(docs[0]['y'], docs[0]['x'])

37.5654604550027 126.909918824353


In [35]:
import pandas as pd
df = pd.read_excel('data/서울아파트전세.xlsx')
df

,NO,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형
0,1,서울특별시 송파구 가락동,헬리오시티,전세,"125,000",송파대로 345,아파트
1,3,서울특별시 송파구 가락동,헬리오시티,전세,"69,000",송파대로 345,아파트
2,5,서울특별시 송파구 가락동,헬리오시티,전세,"110,000",송파대로 345,아파트
3,7,서울특별시 송파구 오금동,송파레미니스,전세,"31,998",성내천로 6,아파트
4,8,서울특별시 송파구 오금동,송파더플래티넘,전세,"56,000",성내천로 50,아파트
...,...,...,...,...,...,...,...
138928,254308,서울특별시 서초구 잠원동,메이플자이,전세,"170,000",잠원로4길 50,아파트
138929,254309,서울특별시 서초구 잠원동,신반포4,전세,"100,000",잠원로 37-48,아파트
138930,254310,서울특별시 서초구 잠원동,동아,전세,"96,000",신반포로33길 15,아파트
138931,254311,서울특별시 서초구 잠원동,동아,전세,"89,200",신반포로33길 15,아파트


In [36]:
df['보증금(만원)'] = df['보증금(만원)'].astype(str).str.replace(',', '').str.strip()
df['보증금(만원)'] = pd.to_numeric(df['보증금(만원)'], errors='coerce')

# 2. 그다음 평균 추가

df['평균금액'] = df.groupby('도로명')['보증금(만원)'].transform('mean')
df.head()

,NO,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,1,서울특별시 송파구 가락동,헬리오시티,전세,125000,송파대로 345,아파트,106050.617992
1,3,서울특별시 송파구 가락동,헬리오시티,전세,69000,송파대로 345,아파트,106050.617992
2,5,서울특별시 송파구 가락동,헬리오시티,전세,110000,송파대로 345,아파트,106050.617992
3,7,서울특별시 송파구 오금동,송파레미니스,전세,31998,성내천로 6,아파트,45455.660550
4,8,서울특별시 송파구 오금동,송파더플래티넘,전세,56000,성내천로 50,아파트,61681.578947


In [37]:
df['평균금액'] = (df['평균금액'].round(0)*10000)
df.head()

,NO,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,1,서울특별시 송파구 가락동,헬리오시티,전세,125000,송파대로 345,아파트,1.060510e+09
1,3,서울특별시 송파구 가락동,헬리오시티,전세,69000,송파대로 345,아파트,1.060510e+09
2,5,서울특별시 송파구 가락동,헬리오시티,전세,110000,송파대로 345,아파트,1.060510e+09
3,7,서울특별시 송파구 오금동,송파레미니스,전세,31998,성내천로 6,아파트,4.545600e+08
4,8,서울특별시 송파구 오금동,송파더플래티넘,전세,56000,성내천로 50,아파트,6.168200e+08


In [38]:
df['평균금액'] = df['평균금액'].astype(int)
df.head()

,NO,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,1,서울특별시 송파구 가락동,헬리오시티,전세,125000,송파대로 345,아파트,1060510000
1,3,서울특별시 송파구 가락동,헬리오시티,전세,69000,송파대로 345,아파트,1060510000
2,5,서울특별시 송파구 가락동,헬리오시티,전세,110000,송파대로 345,아파트,1060510000
3,7,서울특별시 송파구 오금동,송파레미니스,전세,31998,성내천로 6,아파트,454560000
4,8,서울특별시 송파구 오금동,송파더플래티넘,전세,56000,성내천로 50,아파트,616820000


In [39]:
df.to_excel('서울아파트월세최종.xlsx')
print('완료')

완료


In [41]:
import pandas as pd
df = pd.read_excel("data/서울연립월세.xlsx")
df.head()

,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형
0,서울특별시 용산구 후암동,해오름빌,전세,37000,두텁바위로31길 19,다세대
1,서울특별시 강남구 역삼동,더힐,전세,41000,논현로63길 5,다세대
2,서울특별시 중구 신당동,그레이스빌,전세,28700,동호로8나길 2,다세대
3,서울특별시 강남구 역삼동,대하빌딩,전세,21500,강남대로110길 76,다세대
4,서울특별시 강남구 대치동,서울유니빌,전세,26250,삼성로69길 20,다세대


In [42]:
df['평균금액'] = df.groupby('도로명')['보증금(만원)'].transform('mean')
df.head()

,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,서울특별시 용산구 후암동,해오름빌,전세,37000,두텁바위로31길 19,다세대,37000.0
1,서울특별시 강남구 역삼동,더힐,전세,41000,논현로63길 5,다세대,44000.0
2,서울특별시 중구 신당동,그레이스빌,전세,28700,동호로8나길 2,다세대,28700.0
3,서울특별시 강남구 역삼동,대하빌딩,전세,21500,강남대로110길 76,다세대,32600.0
4,서울특별시 강남구 대치동,서울유니빌,전세,26250,삼성로69길 20,다세대,27025.0


In [46]:
df['평균금액'] = (df['평균금액'].round(0)*10000)
df.head()

,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,서울특별시 용산구 후암동,해오름빌,전세,37000,두텁바위로31길 19,다세대,0
1,서울특별시 강남구 역삼동,더힐,전세,41000,논현로63길 5,다세대,0
2,서울특별시 중구 신당동,그레이스빌,전세,28700,동호로8나길 2,다세대,0
3,서울특별시 강남구 역삼동,대하빌딩,전세,21500,강남대로110길 76,다세대,0
4,서울특별시 강남구 대치동,서울유니빌,전세,26250,삼성로69길 20,다세대,0


,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,서울특별시 용산구 후암동,해오름빌,전세,37000,두텁바위로31길 19,다세대,-2147483648
1,서울특별시 강남구 역삼동,더힐,전세,41000,논현로63길 5,다세대,-2147483648
2,서울특별시 중구 신당동,그레이스빌,전세,28700,동호로8나길 2,다세대,-2147483648
3,서울특별시 강남구 역삼동,대하빌딩,전세,21500,강남대로110길 76,다세대,-2147483648
4,서울특별시 강남구 대치동,서울유니빌,전세,26250,삼성로69길 20,다세대,-2147483648


In [47]:
import pandas as pd
df = pd.read_excel("data/서울연립월세.xlsx")
df.head()

,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형
0,서울특별시 용산구 후암동,해오름빌,전세,37000,두텁바위로31길 19,다세대
1,서울특별시 강남구 역삼동,더힐,전세,41000,논현로63길 5,다세대
2,서울특별시 중구 신당동,그레이스빌,전세,28700,동호로8나길 2,다세대
3,서울특별시 강남구 역삼동,대하빌딩,전세,21500,강남대로110길 76,다세대
4,서울특별시 강남구 대치동,서울유니빌,전세,26250,삼성로69길 20,다세대


In [48]:
df['평균금액'] = df.groupby('도로명')['보증금(만원)'].transform('mean')
df.head()

,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,서울특별시 용산구 후암동,해오름빌,전세,37000,두텁바위로31길 19,다세대,37000.0
1,서울특별시 강남구 역삼동,더힐,전세,41000,논현로63길 5,다세대,44000.0
2,서울특별시 중구 신당동,그레이스빌,전세,28700,동호로8나길 2,다세대,28700.0
3,서울특별시 강남구 역삼동,대하빌딩,전세,21500,강남대로110길 76,다세대,32600.0
4,서울특별시 강남구 대치동,서울유니빌,전세,26250,삼성로69길 20,다세대,27025.0


In [49]:
df['평균금액'] = (df['평균금액'].round(0)*10000)
df.head()

,시군구,단지명,전월세구분,보증금(만원),도로명,주택유형,평균금액
0,서울특별시 용산구 후암동,해오름빌,전세,37000,두텁바위로31길 19,다세대,370000000.0
1,서울특별시 강남구 역삼동,더힐,전세,41000,논현로63길 5,다세대,440000000.0
2,서울특별시 중구 신당동,그레이스빌,전세,28700,동호로8나길 2,다세대,287000000.0
3,서울특별시 강남구 역삼동,대하빌딩,전세,21500,강남대로110길 76,다세대,326000000.0
4,서울특별시 강남구 대치동,서울유니빌,전세,26250,삼성로69길 20,다세대,270250000.0


In [50]:
df.to_excel('서울연립전세최종.xlsx')
print('finish')

finish


In [51]:
import requests
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_coord(address):
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        docs = res.json().get("documents", [])
        
        if not docs:
            return pd.Series([None, None])
        
        lat = float(docs[0]['y'])
        lng = float(docs[0]['x'])
        return pd.Series([lat, lng])
    
    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None])

df = pd.read_excel('data/인천서울단독매매가전세최종.xlsx')
df[['위도', '경도']] = df['도로명'].progress_apply(get_coord)
df.to_excel('인천서울단독매매가전세최종1.xlsx', index=False)
print('완료!')

 25%|██████████████████▏                                                      | 36484/146720 [38:16<1:55:39, 15.89it/s]


KeyboardInterrupt: 

In [53]:
import requests
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_coord(address):
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        docs = res.json().get("documents", [])
        
        if not docs:
            return pd.Series([None, None])
        
        lat = float(docs[0]['y'])
        lng = float(docs[0]['x'])
        return pd.Series([lat, lng])
    
    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None])

df = pd.read_excel('data/인천서울단독매매가전세최종.xlsx')
total = len(df)

results = []
for i, row in tqdm(df.iterrows(), total=total, desc="변환 중"):
    result = get_coord(row['도로명'])
    results.append(result)

    # 1000개마다 중간 저장
    if (i + 1) % 1000 == 0:
        temp_df = df.iloc[:i+1].copy()
        temp_df[['위도', '경도']] = pd.DataFrame(results)
        temp_df.to_excel(f'결과_{i+1}개완료.xlsx', index=False)
        print(f"\n✅ {i+1}개 저장 완료!")

df[['위도', '경도']] = pd.DataFrame(results)
df.to_excel('인천서울단독매매가전세_최종.xlsx', index=False)
print('✅ 전체 완료!')

변환 중:   7%|████▍                                                               | 1002/15250 [01:08<22:04, 10.76it/s]


✅ 1000개 저장 완료!


변환 중:  13%|████████▉                                                           | 2001/15250 [02:16<29:56,  7.38it/s]


✅ 2000개 저장 완료!


변환 중:  20%|█████████████▍                                                      | 3002/15250 [03:23<22:24,  9.11it/s]

...


✅ 14000개 저장 완료!


변환 중:  98%|█████████████████████████████████████████████████████████████████▉ | 15003/15250 [17:01<01:33,  2.64it/s]


✅ 15000개 저장 완료!


변환 중: 100%|███████████████████████████████████████████████████████████████████| 15250/15250 [17:17<00:00, 14.70it/s]


✅ 전체 완료!


In [54]:
import requests
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_coord(address):
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        docs = res.json().get("documents", [])
        
        if not docs:
            return pd.Series([None, None])
        
        lat = float(docs[0]['y'])
        lng = float(docs[0]['x'])
        return pd.Series([lat, lng])
    
    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None])

df = pd.read_excel('data/인천서울아파트연립전세최종.xlsx')
total = len(df)

results = []
for i, row in tqdm(df.iterrows(), total=total, desc="변환 중"):
    result = get_coord(row['도로명'])
    results.append(result)

    # 1000개마다 중간 저장
    if (i + 1) % 1000 == 0:
        temp_df = df.iloc[:i+1].copy()
        temp_df[['위도', '경도']] = pd.DataFrame(results)
        temp_df.to_excel(f'결과_{i+1}개완료.xlsx', index=False)
        print(f"\n✅ {i+1}개 저장 완료!")

df[['위도', '경도']] = pd.DataFrame(results)
df.to_excel('인천서울아파트연립전세_최종.xlsx', index=False)
print('✅ 전체 완료!')

변환 중:   2%|█▌                                                                | 1003/43325 [01:05<1:06:53, 10.55it/s]


✅ 1000개 저장 완료!


변환 중:   5%|███                                                               | 2002/43325 [02:11<1:12:52,  9.45it/s]


✅ 2000개 저장 완료!


변환 중:   7%|████▌                                                             | 3000/43325 [03:17<2:08:42,  5.22it/s]

...


✅ 42000개 저장 완료!


변환 중:  99%|██████████████████████████████████████████████████████████████████▍| 43001/43325 [50:30<08:19,  1.54s/it]


✅ 43000개 저장 완료!


변환 중: 100%|███████████████████████████████████████████████████████████████████| 43325/43325 [50:51<00:00, 14.20it/s]


✅ 전체 완료!


In [56]:
import requests
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def get_coord(address):
    try:
        url = "https://dapi.kakao.com/v2/local/search/address.json"
        headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
        res = requests.get(url, headers=headers, params={"query": address}, timeout=5)
        docs = res.json().get("documents", [])
        
        if not docs:
            return pd.Series([None, None])
        
        lat = float(docs[0]['y'])
        lng = float(docs[0]['x'])
        return pd.Series([lat, lng])
    
    except Exception as e:
        print(f"변환 실패: {address} / 에러: {e}")
        return pd.Series([None, None])

df = pd.read_excel('data/인천서울연립아파트매매가최종.xlsx')
total = len(df)

results = []
for i, row in tqdm(df.iterrows(), total=total, desc="변환 중"):
    result = get_coord(row['도로명'])
    results.append(result)

    # 1000개마다 중간 저장
    if (i + 1) % 1000 == 0:
        temp_df = df.iloc[:i+1].copy()
        temp_df[['위도', '경도']] = pd.DataFrame(results)
        temp_df.to_excel(f'결과_{i+1}개완료.xlsx', index=False)
        print(f"\n✅ {i+1}개 저장 완료!")

df[['위도', '경도']] = pd.DataFrame(results)
df.to_excel('인천서울연립아파트매매가_최종.xlsx', index=False)
print('✅ 전체 완료!')

변환 중:   3%|██▏                                                                 | 1003/30832 [01:05<44:48, 11.09it/s]


✅ 1000개 저장 완료!


변환 중:   6%|████▍                                                               | 2002/30832 [02:09<49:26,  9.72it/s]


✅ 2000개 저장 완료!


변환 중:  10%|██████▌                                                             | 3003/30832 [03:13<56:49,  8.16it/s]

...


✅ 29000개 저장 완료!


변환 중:  97%|█████████████████████████████████████████████████████████████████▏ | 30002/30832 [34:37<08:46,  1.58it/s]


✅ 30000개 저장 완료!


변환 중: 100%|███████████████████████████████████████████████████████████████████| 30832/30832 [35:25<00:00, 14.50it/s]


✅ 전체 완료!


In [14]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# 정규화 기준값
HOSPITAL_MAX = 0.02  # 소아과 접근성 최고점
WELFARE_MAX  = 0.03  # 아동센터 접근성 최고점
CLINIC_MAX   = 0.6
# 데이터 로드
df_hospital = pd.read_excel('data/(의료)주변소아과.xlsx')
df_welfare  = pd.read_excel('data/(복지)통합아동지원센터.xlsx')
df_clinic   = pd.read_excel('data/[의료]병원_최종.xlsx')  # 파일명 맞게 수정

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
  
    s_hospital = get_access_score(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)
    n_hospital = count_within(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)

  
    s_welfare = get_access_score(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)
    n_welfare = count_within(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)

   
    s_clinic = get_access_score(my_lat, my_lng, df_clinic, '위도', '경도', 1000)
    n_clinic = count_within(my_lat, my_lng, df_clinic, '위도', '경도', 1000)
    
    # 정규화
    v1 = normalize(s_hospital, 0, HOSPITAL_MAX)
    v2 = normalize(s_welfare,  0, WELFARE_MAX)
    v3 = normalize(s_clinic,   0, CLINIC_MAX)
    # 점수 계산
    score = round((v1 + v2 + v3) / 3 * 100, 2)

     # 결과 출력
    print(f"소아과 개수 (2km):             {n_hospital}개")
    print(f"소아과 접근성 (2km):           {s_hospital:.4f}  → V1: {v1:.4f}")
    print(f"통합아동지원센터 개수 (2km):   {n_welfare}개")
    print(f"통합아동지원센터 접근성 (2km): {s_welfare:.4f}  → V2: {v2:.4f}")
    print(f"추가병원 개수 (2km):           {n_clinic}개")
    print(f"추가병원 접근성 (2km):         {s_clinic:.4f}  → V3: {v3:.4f}")
    print()
    print(f"의료·복지 점수 = (V1+V2+V3)/3 × 100")
    print(f"              = {score}점")

주소를 입력하세요:  인천광역시 부평구 마장로 121


소아과 개수 (2km):             23개
소아과 접근성 (2km):           0.0163  → V1: 0.8164
통합아동지원센터 개수 (2km):   12개
통합아동지원센터 접근성 (2km): 0.0104  → V2: 0.3459
추가병원 개수 (2km):           36개
추가병원 접근성 (2km):         0.0829  → V3: 0.1382

의료·복지 점수 = (V1+V2+V3)/3 × 100
              = 43.35점


,시설명,종류,시도명,시군구명,행정동,주소,경도,위도
0,(사)경찰공제회 강서적성의원,의원,서울,강서구,외발산동,"서울특별시 강서구 남부순환로 171, 강서운전면허시험장만남의광장 적성검사소호 (외발산동)",126.821985,37.549498
1,(사)경찰공제회남부의원,의원,인천,인천남동구,고잔동,"인천광역시 남동구 아암대로 1247, (고잔동)",126.708861,37.384910
2,(사)모퉁이복지재단 인천재활의원,의원,인천,인천미추홀구,주안동,"인천광역시 미추홀구 인주대로 290, (주안동)",126.668643,37.451879
3,(사)인구보건복지협회 서울지회 가족보건의원,의원,서울,광진구,중곡동,"서울특별시 광진구 긴고랑로13길 62, 1층, 2층 (중곡동)",127.083230,37.565080
4,(사)한국건강관리협회 중앙검사본부 중앙검사의원,의원,서울,강서구,등촌동,"서울특별시 강서구 화곡로 372, ND타워 지4,지1, 1~4층 (등촌동)",126.853387,37.556571
...,...,...,...,...,...,...,...,...
32891,힘찬부성정형외과의원,의원,서울,강서구,화곡동,"서울특별시 강서구 곰달래로 223, (화곡동)",126.857783,37.531819
32892,힘찬세상경희한의원,한의원,서울,용산구,한강로2가,"서울특별시 용산구 한강대로 109, (한강로2가, 용성비즈텔)",126.968167,37.530005
32893,힘찬하나로한방병원,한방병원,서울,강서구,화곡동,"서울특별시 강서구 강서로 173, 터보빌딩 7,8,9층 (화곡동)",126.839626,37.541919
32894,힘찬한의원,한의원,인천,인천서구,당하동,"인천광역시 서구 고산후로95번길 20, 힘찬프라자 2층 201호 (당하동)",126.696749,37.592665


In [19]:
from math import radians, sin, cos, sqrt, atan2
import pandas as pd
import requests

KAKAO_API_KEY = "YOUR_KAKAO_API_KEY"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    φ1, φ2 = radians(lat1), radians(lat2)
    Δφ = radians(lat2 - lat1)
    Δλ = radians(lon2 - lon1)
    a = sin(Δφ/2)**2 + cos(φ1)*cos(φ2)*sin(Δλ/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def get_coord(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    res = requests.get(url, headers=headers, params={"query": address})
    docs = res.json().get("documents", [])
    if docs:
        return float(docs[0]['y']), float(docs[0]['x'])
    return None, None

def get_access_score(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    df_filtered = df[df['거리'] <= radius]
    if len(df_filtered) == 0:
        return 0
    return (1 / df_filtered['거리']).sum()

def count_within(my_lat, my_lng, df, lat_col, lng_col, radius):
    df = df.copy()
    df['거리'] = df.apply(
        lambda row: haversine(my_lat, my_lng, row[lat_col], row[lng_col]), axis=1
    )
    return int((df['거리'] <= radius).sum())

def normalize(value, v_min, v_max, reverse=False):
    value = max(v_min, min(v_max, value))
    norm = (value - v_min) / (v_max - v_min)
    return round(1 - norm if reverse else norm, 4)

# 정규화 기준값
HOSPITAL_MAX = 0.01  # 소아과 접근성 최고점
WELFARE_MAX  = 0.01  # 아동센터 접근성 최고점
CLINIC_MAX   = 0.45
# 데이터 로드
df_hospital = pd.read_excel('data/(의료)주변소아과.xlsx')
df_welfare  = pd.read_excel('data/(복지)통합아동지원센터.xlsx')
df_clinic   = pd.read_excel('data/[의료]병원_최종.xlsx')  # 파일명 맞게 수정

# 주소 입력
address = input("주소를 입력하세요: ")
my_lat, my_lng = get_coord(address)

if my_lat is None:
    print("주소를 찾을 수 없어요!")
else:
  
    s_hospital = get_access_score(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)
    n_hospital = count_within(my_lat, my_lng, df_hospital, '좌표(Y)', '좌표(X)', 2000)

  
    s_welfare = get_access_score(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)
    n_welfare = count_within(my_lat, my_lng, df_welfare, 'Y좌표값', 'X좌표값', 2000)

   
    s_clinic = get_access_score(my_lat, my_lng, df_clinic, '위도', '경도', 1000)
    n_clinic = count_within(my_lat, my_lng, df_clinic, '위도', '경도', 1000)
    
    # 정규화
    v1 = normalize(s_hospital, 0, HOSPITAL_MAX)
    v2 = normalize(s_welfare,  0, WELFARE_MAX)
    v3 = normalize(s_clinic,   0, CLINIC_MAX)
    # 점수 계산
    score = round((v1 + v2 + v3) / 3 * 100, 2)

     # 결과 출력
    print(f"소아과 개수 (2km):             {n_hospital}개")
    print(f"소아과 접근성 (2km):           {s_hospital:.4f}  → V1: {v1:.4f}")
    print(f"통합아동지원센터 개수 (2km):   {n_welfare}개")
    print(f"통합아동지원센터 접근성 (2km): {s_welfare:.4f}  → V2: {v2:.4f}")
    print(f"추가병원 개수 (2km):           {n_clinic}개")
    print(f"추가병원 접근성 (2km):         {s_clinic:.4f}  → V3: {v3:.4f}")
    print()
    print(f"의료·복지 점수 = (V1+V2+V3)/3 × 100")
    print(f"              = {score}점")

주소를 입력하세요:  인천광역시 부평구 마장로 121


소아과 개수 (2km):             23개
소아과 접근성 (2km):           0.0163  → V1: 1.0000
통합아동지원센터 개수 (2km):   12개
통합아동지원센터 접근성 (2km): 0.0104  → V2: 1.0000
추가병원 개수 (2km):           36개
추가병원 접근성 (2km):         0.0829  → V3: 0.1843

의료·복지 점수 = (V1+V2+V3)/3 × 100
              = 72.81점
